# Task MP-FC-006: Provisional-v2 Gates and Baseline Scorecard (No ML Models)
**Repository:** `ankit-choubey/Mandi_Prediction`  
**Execution Environment:** Kaggle CPU Kernel (Internet Disabled)  
**Evaluated Dataset:** `theankitchoubey/mandi-prediction-audit-003b` (v1 private, unchanged)  

### Objectives:
1. **Panel v2 Regeneration & Assertion Gate:** Execute Pipeline v2 and assert SHA-256 == `851e718468d6b8c9fd07944535a462338e71aa105af25339fe9438b697088f31`.
2. **Part B1 - Forensic Diagnostic Gates (Report Only):**
   - a. Alias transition metrics: gap_days, overlap_days, overlapping keys, exact-price agreement across all accepted alias pairs, Baripaal vs Varipaal, and Jhansi Grain variants.
   - b. Reclassify 1,895 unmatched in-window Excel rows into explicit categories (quarantine, low-obs, spelling, absent, other) by district and crop.
   - c. Reconcile Tier A training observations (v1: 384,887 vs v2: 404,498 cutoff `2025-12-18`).
   - d. Fractional prices from `2026-03`: distribution of decimals, 5% month threshold, adjacent-day integer prices, and commodity median price ratios.
   - e. Monthly reporting depth: active markets, unique commodities, and varieties across all 61 months.
3. **Part B2 - Baseline Forecasting Scorecard (No ML Models):**
   - Pure functions, As-Of operational constraint (history dated <= origin date $t$).
   - 4 Expanding Folds: F1, F2, F3, F4 with 7-day embargo and origin recency filtering (>= 3 obs in 30 days).
   - Models: Naive, Moving Average (MA), Seasonal Naive, Drift.
   - Target: Modal price in $[t+6, t+8]$ closest to $t+7$ (tie: earlier).
   - Metrics: MAE, RMSE, MAPE, Median AE, Bias, Share within 5%, Abstention rate, Skill vs Naive.
   - Cluster bootstrap over series (1,000 resamples, seed 42) for MAE and Skill 95% CIs.
   - Scorecard tables: Fold x Tier, Fold x Focus Crop, and Sensitivities.


In [ ]:
# Cell 1: Environment, Library Versions, Bootstrap Modules, and Input Discovery
import os
import sys
import glob
import json
import base64
import hashlib
import time
import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import sklearn
import openpyxl

print("=== ENVIRONMENT & LIBRARY VERSIONS ===")
print(f"Python: {sys.version}")
print(f"Platform: {sys.platform}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"scipy: {scipy.__version__}")
print(f"sklearn: {sklearn.__version__}")
print(f"openpyxl: {openpyxl.__version__}")

np.random.seed(42)
print("Random seed initialized to 42.")

working_root = Path("/kaggle/working") if os.path.exists("/kaggle") else Path(".")
src_dir = working_root / "src"
ref_dir = working_root / "data" / "reference"
out005_dir = working_root / "experiments" / "outputs" / "005"
scorecards_dir = working_root / "experiments" / "reports" / "scorecards"

src_dir.mkdir(parents=True, exist_ok=True)
ref_dir.mkdir(parents=True, exist_ok=True)
out005_dir.mkdir(parents=True, exist_ok=True)
scorecards_dir.mkdir(parents=True, exist_ok=True)

if not (src_dir / "__init__.py").exists():
    (src_dir / "__init__.py").write_text("")

if str(working_root.resolve()) not in sys.path:
    sys.path.insert(0, str(working_root.resolve()))
if "/kaggle/working" not in sys.path and os.path.exists("/kaggle"):
    sys.path.insert(0, "/kaggle/working")

b64_payloads = {
    src_dir / "market_identity.py": "IiIiCk1hcmtldCBJZGVudGl0eSBOb3JtYWxpemF0aW9uIGFuZCBFdmlkZW5jZS1CYXNlZCBDYW5vbmljYWxpemF0aW9uIEVuZ2luZS4KClByb3ZpZGVzIGRldGVybWluaXN0aWMsIHB1cmUgZnVuY3Rpb25zIGZvciBBUE1DIG1hbmRpIG1hcmtldCBuYW1lIG5vcm1hbGl6YXRpb24sCmV2aWRlbmNlLWJhc2VkIGFsaWFzIGV2YWx1YXRpb24sIGFsaWFzIHRhYmxlIG1hbmFnZW1lbnQsIGFuZCBEYXRhRnJhbWUKY2Fub25pY2FsaXphdGlvbiB3aXRob3V0IGV4dGVybmFsIEkvTyBzaWRlIGVmZmVjdHMuCiIiIgoKaW1wb3J0IG9zCmltcG9ydCByZQppbXBvcnQgc3RyaW5nCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBEaWN0LCBMaXN0LCBPcHRpb25hbCwgVHVwbGUsIFVuaW9uCmltcG9ydCBwYW5kYXMgYXMgcGQKCgojIFJlcXVpcmVkIHNjaGVtYSBmb3IgZGF0YS9yZWZlcmVuY2UvbWFya2V0X2FsaWFzZXMuY3N2Ck1BUktFVF9BTElBU19DT0xVTU5TOiBMaXN0W3N0cl0gPSBbCiAgICAiZGlzdHJpY3QiLAogICAgIm1hcmtldF9yYXciLAogICAgIm1hcmtldF9jYW5vbmljYWwiLAogICAgInJ1bGUiLAogICAgImV2aWRlbmNlIiwKICAgICJmaXJzdF9kYXRlIiwKICAgICJsYXN0X2RhdGUiLAogICAgIm92ZXJsYXBfZGF5cyIsCiAgICAicHJpY2VfYWdyZWVtZW50IiwKICAgICJzdGF0dXMiCl0KCgpkZWYgbm9ybWFsaXplX21hcmtldF9uYW1lKG5hbWU6IE9wdGlvbmFsW3N0cl0pIC0+IHN0cjoKICAgICIiIgogICAgTm9ybWFsaXplcyBhIG1hcmtldCBuYW1lIGludG8gYSBjYW5vbmljYWwgdG9rZW4gc3RyaW5nOgogICAgLSBMb3dlcmNhc2UKICAgIC0gU3RyaXAgcHVuY3R1YXRpb24gKHJlcGxhY2UgcHVuY3R1YXRpb24gbWFya3Mgd2l0aCB3aGl0ZXNwYWNlKQogICAgLSBDb2xsYXBzZSBtdWx0aXBsZSB3aGl0ZXNwYWNlIGNoYXJhY3RlcnMgaW50byBhIHNpbmdsZSBzcGFjZQogICAgLSBSZW1vdmUgdHJhaWxpbmcgJ2FwbWMnIHdvcmQKICAgIC0gU3RyaXAgbGVhZGluZy90cmFpbGluZyB3aGl0ZXNwYWNlCiAgICAKICAgIEV4YW1wbGVzOgogICAgICAgICdHaGF6aWFiYWQgQVBNQycgLT4gJ2doYXppYWJhZCcKICAgICAgICAnS2FucHVyKEdyYWluKSBBUE1DJyAtPiAna2FucHVyIGdyYWluJwogICAgICAgICdKaGFuc2kgKEdyYWluKScgLT4gJ2poYW5zaSBncmFpbicKICAgICAgICAnSGFwdXInIC0+ICdoYXB1cicKICAgICIiIgogICAgaWYgbm90IGlzaW5zdGFuY2UobmFtZSwgc3RyKToKICAgICAgICByZXR1cm4gIiIKICAgIAogICAgIyAxLiBMb3dlcmNhc2UKICAgIHMgPSBuYW1lLnN0cmlwKCkubG93ZXIoKQogICAgaWYgbm90IHM6CiAgICAgICAgcmV0dXJuICIiCiAgICAKICAgICMgMi4gU3RyaXAgcHVuY3R1YXRpb24gKHJlcGxhY2Ugd2l0aCBzcGFjZXMgc28gJ0thbnB1cihHcmFpbiknIC0+ICdrYW5wdXIgZ3JhaW4nKQogICAgcyA9IHJlLnN1YihyIlsiICsgcmUuZXNjYXBlKHN0cmluZy5wdW5jdHVhdGlvbikgKyByIl0iLCAiICIsIHMpCiAgICAKICAgICMgMy4gQ29sbGFwc2UgbXVsdGlwbGUgd2hpdGVzcGFjZQogICAgcyA9IHJlLnN1YihyIlxzKyIsICIgIiwgcykuc3RyaXAoKQogICAgCiAgICAjIDQuIFJlbW92ZSB0cmFpbGluZyAnYXBtYycgd29yZCBib3VuZGFyeQogICAgcyA9IHJlLnN1YihyIlxiYXBtYyQiLCAiIiwgcykuc3RyaXAoKQogICAgCiAgICAjIDUuIEZpbmFsIHdoaXRlc3BhY2UgY29sbGFwc2UKICAgIHMgPSByZS5zdWIociJccysiLCAiICIsIHMpLnN0cmlwKCkKICAgIAogICAgcmV0dXJuIHMKCgpkZWYgZXZhbHVhdGVfYWxpYXNfZXZpZGVuY2UoCiAgICBkaXN0cmljdF8xOiBzdHIsCiAgICBuYW1lXzE6IHN0ciwKICAgIGZpcnN0X2RhdGVfMTogc3RyLAogICAgbGFzdF9kYXRlXzE6IHN0ciwKICAgIGRpc3RyaWN0XzI6IHN0ciwKICAgIG5hbWVfMjogc3RyLAogICAgZmlyc3RfZGF0ZV8yOiBzdHIsCiAgICBsYXN0X2RhdGVfMjogc3RyLAogICAgb3ZlcmxhcHBpbmdfcmVjb3Jkc19kZjogT3B0aW9uYWxbcGQuRGF0YUZyYW1lXSA9IE5vbmUKKSAtPiBUdXBsZVtib29sLCBEaWN0W3N0ciwgQW55XV06CiAgICAiIiIKICAgIEV2YWx1YXRlcyB3aGV0aGVyIHR3byByYXcgbWFya2V0IG5hbWVzIHNhdGlzZnkgdGhlIHN0cmljdCBFdmlkZW5jZSBSdWxlOgogICAgCiAgICBBY2NlcHQgYSBwYWlyIE9OTFkgaWYgQUxMIG9mIHRoZSBmb2xsb3dpbmcgaG9sZDoKICAgICAgMS4gU2FtZSBkaXN0cmljdDogZGlzdHJpY3RfMS5sb3dlcigpID09IGRpc3RyaWN0XzIubG93ZXIoKQogICAgICAyLiBUaGUgbm9ybWFsaXNlZCBuYW1lcyBhcmUgaWRlbnRpY2FsOgogICAgICAgICBub3JtYWxpemVfbWFya2V0X25hbWUobmFtZV8xKSA9PSBub3JtYWxpemVfbWFya2V0X25hbWUobmFtZV8yKQogICAgICAzLiBBTkQgZWl0aGVyOgogICAgICAgICAoYSkgdGhlIHR3byByYXcgbmFtZXMnIGRhdGUgcmFuZ2VzIG92ZXJsYXAgYnkgPD0gNSBjYWxlbmRhciBkYXlzLCBPUgogICAgICAgICAoYikgb24gb3ZlcmxhcHBpbmcgKGNvbW1vZGl0eSwgdmFyaWV0eSwgZ3JhZGUsIGRhdGUpIHJvd3MsIHRoZQogICAgICAgICAgICAgZXhhY3QtcHJpY2UgYWdyZWVtZW50IGlzID49IDk1JS4KICAgICAgICAgICAgIAogICAgUmV0dXJuczoKICAgICAgICAoaXNfYWNjZXB0ZWQ6IGJvb2wsIGV2aWRlbmNlX2RpY3Q6IGRpY3QpCiAgICAiIiIKICAgICMgMS4gRGlzdHJpY3QgY2hlY2sKICAgIGQxID0gc3RyKGRpc3RyaWN0XzEpLnN0cmlwKCkubG93ZXIoKQogICAgZDIgPSBzdHIoZGlzdHJpY3RfMikuc3RyaXAoKS5sb3dlcigpCiAgICBzYW1lX2Rpc3RyaWN0ID0gKGQxID09IGQyKSBhbmQgYm9vbChkMSkKICAgIAogICAgIyAyLiBOb3JtYWxpc2VkIG5hbWUgY2hlY2sKICAgIG5vcm1fMSA9IG5vcm1hbGl6ZV9tYXJrZXRfbmFtZShuYW1lXzEpCiAgICBub3JtXzIgPSBub3JtYWxpemVfbWFya2V0X25hbWUobmFtZV8yKQogICAgc2FtZV9ub3JtID0gKG5vcm1fMSA9PSBub3JtXzIpIGFuZCBib29sKG5vcm1fMSkKICAgIAogICAgIyAzLiBEYXRlIHJhbmdlIG92ZXJsYXAgY2FsY3VsYXRpb24KICAgIGR0MV9zdGFydCA9IHBkLnRvX2RhdGV0aW1lKGZpcnN0X2RhdGVfMSkKICAgIGR0MV9lbmQgPSBwZC50b19kYXRldGltZShsYXN0X2RhdGVfMSkKICAgIGR0Ml9zdGFydCA9IHBkLnRvX2RhdGV0aW1lKGZpcnN0X2RhdGVfMikKICAgIGR0Ml9lbmQgPSBwZC50b19kYXRldGltZShsYXN0X2RhdGVfMikKICAgIAogICAgb3ZlcmxhcF9zdGFydCA9IG1heChkdDFfc3RhcnQsIGR0Ml9zdGFydCkKICAgIG92ZXJsYXBfZW5kID0gbWluKGR0MV9lbmQsIGR0Ml9lbmQpCiAgICAKICAgIGlmIG92ZXJsYXBfZW5kID49IG92ZXJsYXBfc3RhcnQ6CiAgICAgICAgb3ZlcmxhcF9kYXlzID0gaW50KChvdmVybGFwX2VuZCAtIG92ZXJsYXBfc3RhcnQpLmRheXMpICsgMQogICAgZWxzZToKICAgICAgICBvdmVybGFwX2RheXMgPSAwCiAgICAgICAgCiAgICBwcmljZV9hZ3JlZW1lbnQ6IE9wdGlvbmFsW2Zsb2F0XSA9IE5vbmUKICAgIHJ1bGVfcGFzc2VkOiBPcHRpb25hbFtzdHJdID0gTm9uZQogICAgZXZpZGVuY2Vfc3RyID0gIiIKICAgIHN0YXR1cyA9ICJVTk1BUFBFRC9SRVZJRVciCiAgICAKICAgIGlmIG5vdCBzYW1lX2Rpc3RyaWN0OgogICAgICAgIHJldHVybiBGYWxzZSwgewogICAgICAgICAgICAic3RhdHVzIjogIlVOTUFQUEVEL1JFVklFVyIsCiAgICAgICAgICAgICJydWxlIjogIkZhaWxlZCAoZGlmZmVyZW50IGRpc3RyaWN0cykiLAogICAgICAgICAgICAiZXZpZGVuY2UiOiBmIkRpc3RyaWN0IG1pc21hdGNoOiAne2Rpc3RyaWN0XzF9JyB2cyAne2Rpc3RyaWN0XzJ9JyIsCiAgICAgICAgICAgICJvdmVybGFwX2RheXMiOiBvdmVybGFwX2RheXMsCiAgICAgICAgICAgICJwcmljZV9hZ3JlZW1lbnQiOiBOb25lCiAgICAgICAgfQogICAgICAgIAogICAgaWYgbm90IHNhbWVfbm9ybToKICAgICAgICByZXR1cm4gRmFsc2UsIHsKICAgICAgICAgICAgInN0YXR1cyI6ICJVTk1BUFBFRC9SRVZJRVciLAogICAgICAgICAgICAicnVsZSI6ICJGYWlsZWQgKG5vcm1hbGlzZWQgbmFtZXMgbWlzbWF0Y2gpIiwKICAgICAgICAgICAgImV2aWRlbmNlIjogZiJOb3JtYWxpemVkIG1pc21hdGNoOiAne25vcm1fMX0nIHZzICd7bm9ybV8yfSciLAogICAgICAgICAgICAib3ZlcmxhcF9kYXlzIjogb3ZlcmxhcF9kYXlzLAogICAgICAgICAgICAicHJpY2VfYWdyZWVtZW50IjogTm9uZQogICAgICAgIH0KICAgICAgICAKICAgICMgQ29uZGl0aW9uIChhKTogT3ZlcmxhcCA8PSA1IGRheXMgKGUuZy4gY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uKQogICAgaWYgb3ZlcmxhcF9kYXlzIDw9IDU6CiAgICAgICAgcnVsZV9wYXNzZWQgPSAiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIgogICAgICAgIHN0YXR1cyA9ICJBQ0NFUFRFRCIKICAgICAgICBldmlkZW5jZV9zdHIgPSBmIm92ZXJsYXBfZGF5cz17b3ZlcmxhcF9kYXlzfSA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbikiCiAgICBlbHNlOgogICAgICAgICMgQ29uZGl0aW9uIChiKTogUHJpY2UgYWdyZWVtZW50IG9uIG92ZXJsYXBwaW5nIHJlY29yZHMKICAgICAgICBpZiBvdmVybGFwcGluZ19yZWNvcmRzX2RmIGlzIG5vdCBOb25lIGFuZCBub3Qgb3ZlcmxhcHBpbmdfcmVjb3Jkc19kZi5lbXB0eToKICAgICAgICAgICAgdG90YWxfbWF0Y2hlcyA9IGxlbihvdmVybGFwcGluZ19yZWNvcmRzX2RmKQogICAgICAgICAgICBleGFjdF9tYXRjaGVzID0gaW50KChvdmVybGFwcGluZ19yZWNvcmRzX2RmWyJwcmljZV8xIl0gPT0gb3ZlcmxhcHBpbmdfcmVjb3Jkc19kZlsicHJpY2VfMiJdKS5zdW0oKSkKICAgICAgICAgICAgcHJpY2VfYWdyZWVtZW50ID0gcm91bmQoZmxvYXQoZXhhY3RfbWF0Y2hlcyAvIHRvdGFsX21hdGNoZXMpLCA0KSBpZiB0b3RhbF9tYXRjaGVzID4gMCBlbHNlIDAuMAogICAgICAgICAgICAKICAgICAgICAgICAgaWYgcHJpY2VfYWdyZWVtZW50ID49IDAuOTU6CiAgICAgICAgICAgICAgICBydWxlX3Bhc3NlZCA9ICJSdWxlIEIgKG92ZXJsYXBwaW5nIGNvbmNvcmRhbmNlID49IDk1JSkiCiAgICAgICAgICAgICAgICBzdGF0dXMgPSAiQUNDRVBURUQiCiAgICAgICAgICAgICAgICBldmlkZW5jZV9zdHIgPSBmIm92ZXJsYXBfZGF5cz17b3ZlcmxhcF9kYXlzfSA+IDU7IHByaWNlX2FncmVlbWVudD17cHJpY2VfYWdyZWVtZW50Oi40Zn0gPj0gMC45NSAoe2V4YWN0X21hdGNoZXN9L3t0b3RhbF9tYXRjaGVzfSkiCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBydWxlX3Bhc3NlZCA9ICJGYWlsZWQgUnVsZSBCIChwcmljZSBhZ3JlZW1lbnQgPCA5NSUpIgogICAgICAgICAgICAgICAgc3RhdHVzID0gIlVOTUFQUEVEL1JFVklFVyIKICAgICAgICAgICAgICAgIGV2aWRlbmNlX3N0ciA9IGYib3ZlcmxhcF9kYXlzPXtvdmVybGFwX2RheXN9ID4gNTsgcHJpY2VfYWdyZWVtZW50PXtwcmljZV9hZ3JlZW1lbnQ6LjRmfSA8IDAuOTUgKHtleGFjdF9tYXRjaGVzfS97dG90YWxfbWF0Y2hlc30pIgogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHJ1bGVfcGFzc2VkID0gIkZhaWxlZCBSdWxlIEIgKG5vIG92ZXJsYXAgcHJpY2UgcmVjb3JkcykiCiAgICAgICAgICAgIHN0YXR1cyA9ICJVTk1BUFBFRC9SRVZJRVciCiAgICAgICAgICAgIGV2aWRlbmNlX3N0ciA9IGYib3ZlcmxhcF9kYXlzPXtvdmVybGFwX2RheXN9ID4gNTsgbWlzc2luZyBvdmVybGFwcGluZyBwcmljZSBkYXRhIGZvciB2ZXJpZmljYXRpb24iCiAgICAgICAgICAgIAogICAgaXNfYWNjZXB0ZWQgPSAoc3RhdHVzID09ICJBQ0NFUFRFRCIpCiAgICByZXR1cm4gaXNfYWNjZXB0ZWQsIHsKICAgICAgICAic3RhdHVzIjogc3RhdHVzLAogICAgICAgICJydWxlIjogcnVsZV9wYXNzZWQsCiAgICAgICAgImV2aWRlbmNlIjogZXZpZGVuY2Vfc3RyLAogICAgICAgICJvdmVybGFwX2RheXMiOiBvdmVybGFwX2RheXMsCiAgICAgICAgInByaWNlX2FncmVlbWVudCI6IHByaWNlX2FncmVlbWVudAogICAgfQoKCmRlZiBsb2FkX21hcmtldF9hbGlhc2VzKGNzdl9wYXRoOiBVbmlvbltzdHIsIFBhdGhdKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiIKICAgIExvYWRzIGFuZCB2YWxpZGF0ZXMgdGhlIG1hcmtldCBhbGlhcyByZWZlcmVuY2UgdGFibGUgZnJvbSBDU1YuCiAgICBFbnN1cmVzIGFsbCByZXF1aXJlZCBjb2x1bW5zIGFyZSBwcmVzZW50IGFuZCBzdHJpcHMgd2hpdGVzcGFjZS4KICAgICIiIgogICAgcGF0aF9vYmogPSBQYXRoKGNzdl9wYXRoKQogICAgaWYgbm90IHBhdGhfb2JqLmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiTWFya2V0IGFsaWFzZXMgcmVmZXJlbmNlIGZpbGUgbm90IGZvdW5kOiB7cGF0aF9vYmp9IikKICAgICAgICAKICAgIGRmID0gcGQucmVhZF9jc3YocGF0aF9vYmopCiAgICBtaXNzaW5nX2NvbHMgPSBbYyBmb3IgYyBpbiBNQVJLRVRfQUxJQVNfQ09MVU1OUyBpZiBjIG5vdCBpbiBkZi5jb2x1bW5zXQogICAgaWYgbWlzc2luZ19jb2xzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJNYXJrZXQgYWxpYXNlcyBDU1YgaXMgbWlzc2luZyByZXF1aXJlZCBjb2x1bW5zOiB7bWlzc2luZ19jb2xzfSIpCiAgICAgICAgCiAgICAjIENsZWFuIHdoaXRlc3BhY2Ugb24gc3RyaW5nIGNvbHVtbnMKICAgIGZvciBjIGluIFsiZGlzdHJpY3QiLCAibWFya2V0X3JhdyIsICJtYXJrZXRfY2Fub25pY2FsIiwgInJ1bGUiLCAic3RhdHVzIl06CiAgICAgICAgaWYgYyBpbiBkZi5jb2x1bW5zOgogICAgICAgICAgICBkZltjXSA9IGRmW2NdLmFzdHlwZShzdHIpLnN0ci5zdHJpcCgpCiAgICAgICAgICAgIAogICAgcmV0dXJuIGRmCgoKZGVmIGJ1aWxkX2FsaWFzX21hcHBpbmcoYWxpYXNfZGY6IHBkLkRhdGFGcmFtZSkgLT4gRGljdFtUdXBsZVtzdHIsIHN0cl0sIHN0cl06CiAgICAiIiIKICAgIEJ1aWxkcyBhIGxvb2t1cCBtYXBwaW5nIGZyb20gYWNjZXB0ZWQgbWFya2V0IGFsaWFzZXM6CiAgICAgICAgKGRpc3RyaWN0X2xvd2VyLCByYXdfbWFya2V0X2xvd2VyKSAtPiBjYW5vbmljYWxfbWFya2V0X25hbWUKICAgIE9ubHkgcmVjb3JkcyB3aXRoIHN0YXR1cyA9PSAnQUNDRVBURUQnIGFyZSBpbmNsdWRlZC4KICAgICIiIgogICAgbWFwcGluZzogRGljdFtUdXBsZVtzdHIsIHN0cl0sIHN0cl0gPSB7fQogICAgYWNjZXB0ZWRfZGYgPSBhbGlhc19kZlthbGlhc19kZlsic3RhdHVzIl0uc3RyLnVwcGVyKCkgPT0gIkFDQ0VQVEVEIl0KICAgIAogICAgZm9yIF8sIHJvdyBpbiBhY2NlcHRlZF9kZi5pdGVycm93cygpOgogICAgICAgIGtleSA9IChyb3dbImRpc3RyaWN0Il0uc3RyaXAoKS5sb3dlcigpLCByb3dbIm1hcmtldF9yYXciXS5zdHJpcCgpLmxvd2VyKCkpCiAgICAgICAgbWFwcGluZ1trZXldID0gcm93WyJtYXJrZXRfY2Fub25pY2FsIl0uc3RyaXAoKQogICAgICAgIAogICAgcmV0dXJuIG1hcHBpbmcKCgpkZWYgYXBwbHlfbWFya2V0X2Nhbm9uaWNhbGl6YXRpb24oCiAgICBkZjogcGQuRGF0YUZyYW1lLAogICAgYWxpYXNfc291cmNlOiBVbmlvbltwZC5EYXRhRnJhbWUsIERpY3RbVHVwbGVbc3RyLCBzdHJdLCBzdHJdLCBzdHIsIFBhdGhdLAogICAgZGlzdHJpY3RfY29sOiBzdHIgPSAiRGlzdHJpY3QiLAogICAgbWFya2V0X2NvbDogc3RyID0gIk1hcmtldCIsCiAgICByYXdfbWFya2V0X2NvbDogc3RyID0gIk1hcmtldF9SYXciCikgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiCiAgICBBcHBsaWVzIGV2aWRlbmNlLWJhc2VkIG1hcmtldCBjYW5vbmljYWxpemF0aW9uIHRvIGEgRGF0YUZyYW1lOgogICAgMS4gUHJlc2VydmVzIG9yaWdpbmFsIG1hcmtldCBuYW1lIGluIGByYXdfbWFya2V0X2NvbGAgKGUuZy4gJ01hcmtldF9SYXcnKS4KICAgIDIuIE1hcHMgYG1hcmtldF9jb2xgIHRvIGBtYXJrZXRfY2Fub25pY2FsYCB1c2luZyB0aGUgYWNjZXB0ZWQgYWxpYXMgbWFwcGluZy4KICAgIDMuIElmIG5vIGFjY2VwdGVkIGFsaWFzIGlzIGZvdW5kLCBsZWF2ZXMgYG1hcmtldF9jb2xgIGlkZW50aWNhbCB0byBgcmF3X21hcmtldF9jb2xgLgogICAgCiAgICBQdXJlIGZ1bmN0aW9uOiByZXR1cm5zIGEgbmV3IERhdGFGcmFtZSBjb3B5LgogICAgIiIiCiAgICBvdXRfZGYgPSBkZi5jb3B5KCkKICAgIAogICAgIyAxLiBQcmVzZXJ2ZSBvcmlnaW5hbCBuYW1lCiAgICBpZiByYXdfbWFya2V0X2NvbCBub3QgaW4gb3V0X2RmLmNvbHVtbnM6CiAgICAgICAgb3V0X2RmW3Jhd19tYXJrZXRfY29sXSA9IG91dF9kZlttYXJrZXRfY29sXQogICAgICAgIAogICAgIyAyLiBSZXNvbHZlIG1hcHBpbmcKICAgIGlmIGlzaW5zdGFuY2UoYWxpYXNfc291cmNlLCBkaWN0KToKICAgICAgICBtYXBwaW5nID0gYWxpYXNfc291cmNlCiAgICBlbGlmIGlzaW5zdGFuY2UoYWxpYXNfc291cmNlLCBwZC5EYXRhRnJhbWUpOgogICAgICAgIG1hcHBpbmcgPSBidWlsZF9hbGlhc19tYXBwaW5nKGFsaWFzX3NvdXJjZSkKICAgIGVsc2U6CiAgICAgICAgYWxpYXNfZGYgPSBsb2FkX21hcmtldF9hbGlhc2VzKGFsaWFzX3NvdXJjZSkKICAgICAgICBtYXBwaW5nID0gYnVpbGRfYWxpYXNfbWFwcGluZyhhbGlhc19kZikKICAgICAgICAKICAgICMgMy4gQ2Fub25pY2FsaXplIE1hcmtldCBjb2x1bW4KICAgIGRlZiBfbWFwX3Jvdyhyb3cpOgogICAgICAgIGRpc3QgPSBzdHIocm93W2Rpc3RyaWN0X2NvbF0pLnN0cmlwKCkubG93ZXIoKQogICAgICAgIHJhd19ta3QgPSBzdHIocm93W3Jhd19tYXJrZXRfY29sXSkuc3RyaXAoKS5sb3dlcigpCiAgICAgICAgcmV0dXJuIG1hcHBpbmcuZ2V0KChkaXN0LCByYXdfbWt0KSwgcm93W3Jhd19tYXJrZXRfY29sXSkKICAgICAgICAKICAgIG91dF9kZlttYXJrZXRfY29sXSA9IG91dF9kZi5hcHBseShfbWFwX3JvdywgYXhpcz0xKQogICAgcmV0dXJuIG91dF9kZgoK",
    src_dir / "data_pipeline.py": "...",
    src_dir / "ingest_agmarknet_arrivals.py": "...",
    src_dir / "baselines.py": "...",
    ref_dir / "market_aliases.csv": "ZGlzdHJpY3QsbWFya2V0X3JhdyxtYXJrZXRfY2Fub25pY2FsLHJ1bGUsZXZpZGVuY2UsZmlyc3RfZGF0ZSxsYXN0X2RhdGUsb3ZlcmxhcF9kYXlzLHByaWNlX2FncmVlbWVudCxzdGF0dXMKR2hhemlhYmFkLEdoYXppYWJhZCxHaGF6aWFiYWQsSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMS0wOS0wMSwyMDI1LTExLTA0LDAsLEFDQ0VQVEVECkdoYXppYWJhZCxHaGF6aWFiYWQgQVBNQyxHaGF6aWFiYWQsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTQgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMDEsMjAyNi0wOS0zMCw0LCxBQ0NFUFRFRApHaGF6aWFiYWQsSGFwdXIsSGFwdXIsSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMS0wOS0wMSwyMDI1LTExLTA1LDAsLEFDQ0VQVEVECkdoYXppYWJhZCxIYXB1ciBBUE1DLEhhcHVyLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz01IDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNSwsQUNDRVBURUQKR2hhemlhYmFkLE11cmFkbmFnYXIsTXVyYWRuYWdhcixJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKR2hhemlhYmFkLE11cmFkbmFnYXIgQVBNQyxNdXJhZG5hZ2FyLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz00IDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNCwsQUNDRVBURUQKR2hhemlhYmFkLE5vaWRhLE5vaWRhLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDIsMjAyNS0xMS0wMywwLCxBQ0NFUFRFRApHaGF6aWFiYWQsTm9pZGEgQVBNQyxOb2lkYSwiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9MyA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0wMSwyMDI2LTA5LTMwLDMsLEFDQ0VQVEVECkpoYW5zaSxCYXJ1d2FzYWdhcixCYXJ1d2FzYWdhcixJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAyLDIwMjUtMTEtMDUsMCwsQUNDRVBURUQKSmhhbnNpLEJhcnV3YXNhZ2FyIEFQTUMsQmFydXdhc2FnYXIsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTMgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMDMsMjAyNi0wOS0zMCwzLCxBQ0NFUFRFRApKaGFuc2ksQ2hpcmdhb24sQ2hpcmdhb24sSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMS0wOS0wMSwyMDI1LTExLTA0LDAsLEFDQ0VQVEVECkpoYW5zaSxDaGlyZ2FvbiBBUE1DLENoaXJnYW9uLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz0zIDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAyLDIwMjYtMDktMzAsMywsQUNDRVBURUQKSmhhbnNpLEd1cnVzYXJhaSxHdXJ1c2FyYWksSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMS0wOS0wMSwyMDI1LTExLTA0LDAsLEFDQ0VQVEVECkpoYW5zaSxHdXJ1c2FyYWkgQVBNQyxHdXJ1c2FyYWksIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTQgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMDEsMjAyNi0wOS0zMCw0LCxBQ0NFUFRFRApKaGFuc2ksSmhhbnNpLEpoYW5zaSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKSmhhbnNpLEpoYW5zaSBBUE1DLEpoYW5zaSwiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9NCA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0wMSwyMDI2LTA5LTMwLDQsLEFDQ0VQVEVECkpoYW5zaSxKaGFuc2kgKEdyYWluKSxKaGFuc2kgKEdyYWluKSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDI0LTA1LTI1LDIwMjUtMTEtMDMsMCwsQUNDRVBURUQKSmhhbnNpLEpoYW5zaSAoR3JhaW4pIEFQTUMsSmhhbnNpIChHcmFpbiksIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTIgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMDIsMjAyNi0wMS0xNCwyLCxBQ0NFUFRFRApKaGFuc2ksSmhhbnNpIEdyYWluIEFQTUMsSmhhbnNpIChHcmFpbiksIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTAgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjYtMDEtMDEsMjAyNi0wMS0xNCwwLCxBQ0NFUFRFRApKaGFuc2ksTWF1cmFuaXB1cixNYXVyYW5pcHVyLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRApKaGFuc2ksTWF1cmFuaXB1ciBBUE1DLE1hdXJhbmlwdXIsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTQgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMDEsMjAyNi0wOS0zMCw0LCxBQ0NFUFRFRApKaGFuc2ksTW90aCxNb3RoLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRApKaGFuc2ksTW90aCBBUE1DLE1vdGgsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTQgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMDEsMjAyNi0wOS0zMCw0LCxBQ0NFUFRFRApLYW5wdXIsQmFyaXBhYWwgQVBNQyxCYXJpcGFhbCBBUE1DLElkZW50aXR5IChzaW5nbGUgdG9rZW4pLFVuaXF1ZSBtYXJrZXQgZm9yIG5vcm1hbGl6ZWQgdG9rZW4gaW4gZGlzdHJpY3QsMjAyNi0wMy0wMSwyMDI2LTA5LTMwLDAsLEFDQ0VQVEVECkthbnB1cixDaG91YmVwdXIsQ2hvdWJlcHVyLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRApLYW5wdXIsQ2hvdWJlcHVyIEFQTUMsQ2hvdWJlcHVyLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz00IDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNCwsQUNDRVBURUQKS2FucHVyLEthbnB1cihHcmFpbiksS2FucHVyKEdyYWluKSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKS2FucHVyLEthbnB1cihHcmFpbikgQVBNQyxLYW5wdXIoR3JhaW4pLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz00IDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNCwsQUNDRVBURUQKS2FucHVyLFV0dGFyaXB1cmEsVXR0YXJpcHVyYSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTEwLTI0LDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKS2FucHVyLFV0dGFyaXB1cmEgQVBNQyxVdHRhcmlwdXJhLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz00IDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNCwsQUNDRVBURUQKS2FucHVyLFZhcmlwYWFsLFZhcmlwYWFsLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRApLYW5wdXIsVmFyaXBhYWwgQVBNQyxWYXJpcGFhbCwiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9NCA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0wMSwyMDI2LTA0LTI2LDQsLEFDQ0VQVEVECkx1Y2tub3csQmFudGhhcmEsQmFudGhhcmEsSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMi0wNS0xOCwyMDI1LTExLTA0LDAsLEFDQ0VQVEVECkx1Y2tub3csQmFudGhhcmEgQVBNQyxCYW50aGFyYSwiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9NCA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0wMSwyMDI2LTA5LTMwLDQsLEFDQ0VQVEVECkx1Y2tub3csTHVja25vdyxMdWNrbm93LElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRApMdWNrbm93LEx1Y2tub3cgQVBNQyxMdWNrbm93LCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz0zIDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAyLDIwMjYtMDktMzAsMywsQUNDRVBURUQKVW5uYW8sQmFuZ2FybWF1LEJhbmdhcm1hdSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKVW5uYW8sQmFuZ2FybWF1IEFQTUMsQmFuZ2FybWF1LCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz00IDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNCwsQUNDRVBURUQKVW5uYW8sUHVyd2EsUHVyd2EsSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMS0wOS0wMSwyMDI1LTA3LTMxLDAsLEFDQ0VQVEVEClVubmFvLFB1cndhIEFQTUMsUHVyd2EsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTAgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjYtMDItMjEsMjAyNi0wOS0zMCwwLCxBQ0NFUFRFRApVbm5hbyxVbm5hbyxVbm5hbyxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKVW5uYW8sVW5uYW8gQVBNQyxVbm5hbywiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9NCA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0wMSwyMDI2LTA5LTMwLDQsLEFDQ0VQVEVECkthbnB1cixCYXJpcGFhbCBBUE1DLFZhcmlwYWFsLFJldmlldyBQZW5kaW5nIChjcm9zcy1waG9uZXRpYyBiL3YpLE5vcm1hbGl6ZWQgbWlzbWF0Y2ggKGJhcmlwYWFsIHZzIHZhcmlwYWFsKTsgY2FuZGlkYXRlIGZvciBtYW51YWwgdmVyaWZpY2F0aW9uLDIwMjYtMDMtMDEsMjAyNi0wOS0zMCwwLCxVTk1BUFBFRC9SRVZJRVcK",
    out005_dir / "diff_77_deep_analysis.csv": "RGlzdHJpY3RfU3RkLE1hcmtldCxDb21tb2RpdHlfU3RkLFZhcmlldHksR3JhZGUsQXJyaXZhbF9EYXRlLE1vZGFsX1ByaWNlX0NsZWFuX2V4Y2VsLE1vZGFsX1ByaWNlX0NsZWFuX3BhbmVsLHByaWNlX2RpZmYscmF0aW8sZXhjZWxfaGFzX2RlY2ltYWxzLHBhbmVsX2hhc19kZWNpbWFscyxQcmljZV9Vbml0LEFycml2YWxfVW5pdF9Tb3VyY2UKR2hhemlhYmFkLE5vaWRhIEFQTUMsT25pb24sT3RoZXIsRkFRLDIwMjYtMDMtMjUsMTc0Ni40NiwxNzQ1LjU5LDAuODcwMDAwMDAwMDAwMTE4MiwxLjAwMDQ5ODM5ODgyMjE3NDcsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpHaGF6aWFiYWQsTm9pZGEgQVBNQyxPbmlvbixPdGhlcixGQVEsMjAyNi0wMy0xNiwxNjIwLjQsMTYwMC4wLDIwLjQwMDAwMDAwMDAwMDA5LDEuMDEyNzUsVHJ1ZSxGYWxzZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLE5vaWRhIEFQTUMsT25pb24sT3RoZXIsRkFRLDIwMjYtMDMtMTIsMTczMy41NCwxNzMxLjA3LDIuNDcwMDAwMDAwMDAwMDI3MywxLjAwMTQyNjg2MzE1NDAwMyxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxOb2lkYSBBUE1DLE9uaW9uLE90aGVyLEZBUSwyMDI2LTAyLTI3LDE2NTIuNzQsMTY3OS44MSwtMjcuMDY5OTk5OTk5OTk5OTM2LDAuOTgzODg1MDgyMjQxNDQ0LFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLE5vaWRhIEFQTUMsT25pb24sT3RoZXIsRkFRLDIwMjYtMDItMjUsMTgwNC44MywxOTE1LjU1LC0xMTAuNzIwMDAwMDAwMDAwMDMsMC45NDIxOTkzNjgzMjc2MzQzLFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKS2FucHVyLEthbnB1cihHcmFpbikgQVBNQyxPbmlvbixPdGhlcixGQVEsMjAyNi0wMy0yNSwxMDQ0LjI1LDEwMDAuMCw0NC4yNSwxLjA0NDI1LFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkthbnB1cixLYW5wdXIoR3JhaW4pIEFQTUMsT25pb24sT3RoZXIsRkFRLDIwMjYtMDMtMTIsMTAyMi4yLDEwMDAuMCwyMi4yMDAwMDAwMDAwMDAwNDUsMS4wMjIyLFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkthbnB1cixLYW5wdXIoR3JhaW4pIEFQTUMsT25pb24sT3RoZXIsRkFRLDIwMjYtMDItMjcsMTIyNC42NywxMjE0Ljc4LDkuODkwMDAwMDAwMDAwMSwxLjAwODE0MTM5MTg1Njk2MixUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkx1Y2tub3csTHVja25vdyBBUE1DLE9uaW9uLE90aGVyLEZBUSwyMDI2LTAzLTI1LDEwNzcuODEsMTA3Ny4wMywwLjc3OTk5OTk5OTk5OTk3MjcsMS4wMDA3MjQyMTM4MTAyMDAyLFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKTHVja25vdyxMdWNrbm93IEFQTUMsT25pb24sT3RoZXIsRkFRLDIwMjYtMDItMjcsMTIwNC45NSwxMjAwLjAsNC45NTAwMDAwMDAwMDAwNDU1LDEuMDA0MTI1LFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxNdXJhZG5hZ2FyIEFQTUMsVG9tYXRvLE90aGVyLEZBUSwyMDI2LTAyLTI3LDIwNDUuMjksMjAwMC4wLDQ1LjI4OTk5OTk5OTk5OTk2NCwxLjAyMjY0NSxUcnVlLEZhbHNlLFJzLi9RdWludGFsLFRvbm5lcwpHaGF6aWFiYWQsTm9pZGEgQVBNQyxUb21hdG8sT3RoZXIsRkFRLDIwMjYtMDMtMjYsMTYzNC43NCwxNjA1LjEzLDI5LjYwOTk5OTk5OTk5OTksMS4wMTg0NDcxMDM5NzI4ODY4LFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLE5vaWRhIEFQTUMsVG9tYXRvLE90aGVyLEZBUSwyMDI2LTAzLTI1LDE4ODMuNjgsMTg4MC44NCwyLjg0MDAwMDAwMDAwMDE0NTUsMS4wMDE1MDk5NjM2MzMyNzAzLFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLE5vaWRhIEFQTUMsVG9tYXRvLE90aGVyLEZBUSwyMDI2LTAzLTEyLDIwMDAuNzUsMTYxOS41MiwzODEuMjMsMS4yMzUzOTY5MDc3MjU3NDU4LFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKTHVja25vdyxMdWNrbm93IEFQTUMsVG9tYXRvLE90aGVyLEZBUSwyMDI2LTAyLTI3LDExMDcuNzQsMTE3OS4yNCwtNzEuNSwwLjkzOTM2NzcyODM2NzQyMzEsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpKaGFuc2ksQ2hpcmdhb24gQVBNQyxXaGVhdCxPdGhlcixGQVEsMjAyNi0wMi0yNywyNDI2LjU1LDI0MjUuMCwxLjU1MDAwMDAwMDAwMDE4MiwxLjAwMDYzOTE3NTI1NzczMixUcnVlLEZhbHNlLFJzLi9RdWludGFsLFRvbm5lcwpKaGFuc2ksSmhhbnNpIEFQTUMsV2hlYXQsT3RoZXIsRkFRLDIwMjYtMDItMjcsMjUwMC4yMiwyNTAwLjQsLTAuMTgwMDAwMDAwMDAwMjkxMDQsMC45OTk5MjgwMTE1MTgxNTcsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpKaGFuc2ksTWF1cmFuaXB1ciBBUE1DLFdoZWF0LE90aGVyLEZBUSwyMDI2LTAyLTI3LDIzNTkuNjksMjQ1MC4wLC05MC4zMDk5OTk5OTk5OTk5NSwwLjk2MzEzODc3NTUxMDIwNDEsVHJ1ZSxGYWxzZSxScy4vUXVpbnRhbCxUb25uZXMKS2FucHVyLEthbnB1cihHcmFpbikgQVBNQyxXaGVhdCxEYXJhLEZBUSwyMDI2LTAyLTI3LDI0NTMuOCwyNDQxLjk3LDExLjgzMDAwMDAwMDAwMDM4MiwxLjAwNDg0NDQ0OTM1ODUwOTgsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpLYW5wdXIsS2FucHVyKEdyYWluKSBBUE1DLFdoZWF0LE1lZGl1bSxGQVEsMjAyNi0wMi0yNywyNDY2LjY0LDIzODAuMCw4Ni42Mzk5OTk5OTk5OTk4NywxLjAzNjQwMzM2MTM0NDUzNzgsVHJ1ZSxGYWxzZSxScy4vUXVpbnRhbCxUb25uZXMKTHVja25vdyxCYW50aGFyYSBBUE1DLFdoZWF0LERhcmEsRkFRLDIwMjYtMDItMjgsMjU2OC4wNiwyNDU4LjAsMTEwLjA1OTk5OTk5OTk5OTk1LDEuMDQ0Nzc2MjQwODQ2MjE2NSxUcnVlLEZhbHNlLFJzLi9RdWludGFsLFRvbm5lcwpMdWNrbm93LEJhbnRoYXJhIEFQTUMsV2hlYXQsRGFyYSxGQVEsMjAyNi0wMi0yNywyNDY3LjMsMjQyNS45LDQxLjQwMDAwMDAwMDAwMDA5LDEuMDE3MDY1ODMxMjM3ODkxLFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKTHVja25vdyxCYW50aGFyYSBBUE1DLFdoZWF0LERhcmEsRkFRLDIwMjYtMDItMjYsMjM2MC42MiwyNDQ2LjAsLTg1LjM4MDAwMDAwMDAwMDExLDAuOTY1MDk0MDMxMDcxMTM2NixUcnVlLEZhbHNlLFJzLi9RdWludGFsLFRvbm5lcwpMdWNrbm93LEJhbnRoYXJhIEFQTUMsV2hlYXQsRGFyYSxGQVEsMjAyNi0wMi0yNSwyNDU1LjkyLDI1MjYuMCwtNzAuMDc5OTk5OTk5OTk5OTMsMC45NzIyNTY1MzIwNjY1MDgzLFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkx1Y2tub3csQmFudGhhcmEgQVBNQyxXaGVhdCxEYXJhLEZBUSwyMDI2LTAyLTI0LDI1MzcuNzcsMjQxNC4wLDEyMy43Njk5OTk5OTk5OTk5OCwxLjA1MTI3MTc0ODEzNTg3NCxUcnVlLEZhbHNlLFJzLi9RdWludGFsLFRvbm5lcwpMdWNrbm93LEJhbnRoYXJhIEFQTUMsV2hlYXQsRGFyYSxGQVEsMjAyNi0wMi0yMywyNDEyLjM0LDI0NjQuMCwtNTEuNjU5OTk5OTk5OTk5ODU0LDAuOTc5MDM0MDkwOTA5MDkxLFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkx1Y2tub3csQmFudGhhcmEgQVBNQyxXaGVhdCxEYXJhLEZBUSwyMDI2LTAyLTIyLDI0NjguMzcsMjQ3NC4wLC01LjYzMDAwMDAwMDAwMDEwOSwwLjk5NzcyNDMzMzA2Mzg2NDEsVHJ1ZSxGYWxzZSxScy4vUXVpbnRhbCxUb25uZXMKVW5uYW8sQmFuZ2FybWF1IEFQTUMsV2hlYXQsRGFyYSxGQVEsMjAyNi0wMi0yNywyMjUwLjEsMjI1MC4xOCwtMC4wNzk5OTk5OTk5OTk5MjcyNCwwLjk5OTk2NDQ0NzI4ODY2MTQsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpVbm5hbyxCYW5nYXJtYXUgQVBNQyxXaGVhdCxEYXJhLEZBUSwyMDI2LTAyLTI1LDIyOTAuOTksMjI4NS4zMSw1LjY3OTk5OTk5OTk5OTgzNiwxLjAwMjQ4NTQzOTYxMjEzMTQsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpHaGF6aWFiYWQsR2hhemlhYmFkIEFQTUMsUmljZSxDb21tb24sRkFRLDIwMjYtMDMtMjksNTQzOS4wOCw1MzQxLjc4LDk3LjMwMDAwMDAwMDAwMDE4LDEuMDE4MjE0OTAyMTQ4NzIyLFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLEdoYXppYWJhZCBBUE1DLFJpY2UsT3RoZXIsRkFRLDIwMjYtMDMtMjQsMTE5MDQuNjcsNzU2Mi4yNSw0MzQyLjQyLDEuNTc0MjIzMjgwMTA4NDMzMyxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxOb2lkYSBBUE1DLFJpY2UsT3RoZXIsRkFRLDIwMjYtMDMtMzAsMzA4My45LDExNzE4LjAsLTg2MzQuMSwwLjI2MzE3NjMwOTk1MDUwMzUsVHJ1ZSxGYWxzZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLE5vaWRhIEFQTUMsUmljZSxPdGhlcixGQVEsMjAyNi0wMy0xMyw3NTU4LjA2LDUzMDAuMCwyMjU4LjA2MDAwMDAwMDAwMDQsMS40MjYwNDkwNTY2MDM3NzM2LFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxOb2lkYSBBUE1DLFJpY2UsT3RoZXIsRkFRLDIwMjYtMDMtMTIsNjA2MC4xNSw0ODU5LjUyLDEyMDAuNjI5OTk5OTk5OTk5MiwxLjI0NzA2NzYxMTYxNTk2MTgsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpLYW5wdXIsS2FucHVyKEdyYWluKSBBUE1DLFJpY2UsT3RoZXIsRkFRLDIwMjYtMDMtMTYsNDMwMy42MywzNjM3LjU4LDY2Ni4wNTAwMDAwMDAwMDAyLDEuMTgzMTAyNTAyMjEzMDA5NyxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkthbnB1cixLYW5wdXIoR3JhaW4pIEFQTUMsUmljZSxDb21tb24sRkFRLDIwMjYtMDItMjcsNDM1NC4xMiwyNzM1LjAsMTYxOS4xMiwxLjU5MTk5OTk5OTk5OTk5OTksVHJ1ZSxGYWxzZSxScy4vUXVpbnRhbCxUb25uZXMKTHVja25vdyxCYW50aGFyYSBBUE1DLFJpY2UsQ29tbW9uLEZBUSwyMDI2LTAzLTI5LDI1MzkuNjgsMjIxNy4yOSwzMjIuMzg5OTk5OTk5OTk5OSwxLjE0NTM5ODIxMTMzMDA0NyxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkx1Y2tub3csQmFudGhhcmEgQVBNQyxSaWNlLENvbW1vbixGQVEsMjAyNi0wMy0yOCwyMjA1LjQxLDIwMDEuNywyMDMuNzA5OTk5OTk5OTk5OCwxLjEwMTc2ODQ5Njc3NzczODgsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpMdWNrbm93LEJhbnRoYXJhIEFQTUMsUmljZSxDb21tb24sRkFRLDIwMjYtMDMtMjcsMjc2OC4zOSwyMjAxLjUyLDU2Ni44Njk5OTk5OTk5OTk5LDEuMjU3NDkwMjc5NDQzMjkzNixUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkx1Y2tub3csQmFudGhhcmEgQVBNQyxSaWNlLENvbW1vbixGQVEsMjAyNi0wMy0yNCw0ODY0LjY4LDIwMDAuMCwyODY0LjY4MDAwMDAwMDAwMDMsMi40MzIzNCxUcnVlLEZhbHNlLFJzLi9RdWludGFsLFRvbm5lcwpMdWNrbm93LEJhbnRoYXJhIEFQTUMsUmljZSxDb21tb24sRkFRLDIwMjYtMDMtMTgsNjM5MC42OCwyODA1LjAsMzU4NS42ODAwMDAwMDAwMDAzLDIuMjc4MzE3MjkwNTUyNTg0NixUcnVlLEZhbHNlLFJzLi9RdWludGFsLFRvbm5lcwpMdWNrbm93LEJhbnRoYXJhIEFQTUMsUmljZSxDb21tb24sRkFRLDIwMjYtMDMtMTYsNTgxOC42NywzNzQyLjAsMjA3Ni42NywxLjU1NDk2MjU4Njg1MTk1MDgsVHJ1ZSxGYWxzZSxScy4vUXVpbnRhbCxUb25uZXMKTHVja25vdyxCYW50aGFyYSBBUE1DLFJpY2UsQ29tbW9uLEZBUSwyMDI2LTAzLTA3LDI0NzkuMjgsMjQxNS4zLDYzLjk4MDAwMDAwMDAwMDAyLDEuMDI2NDg5NDYzMDA2NjY1OSxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkx1Y2tub3csQmFudGhhcmEgQVBNQyxSaWNlLE90aGVyLEZBUSwyMDI2LTAzLTAyLDU2MjYuNjgsMjQwMC4wLDMyMjYuNjgwMDAwMDAwMDAwMywyLjM0NDQ1LFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkx1Y2tub3csQmFudGhhcmEgQVBNQyxSaWNlLE90aGVyLEZBUSwyMDI2LTAyLTI4LDU5MzEuOTgsMjMyNy4wLDM2MDQuOTc5OTk5OTk5OTk5NiwyLjU0OTE5NjM5MDIwMTk3NjcsVHJ1ZSxGYWxzZSxScy4vUXVpbnRhbCxUb25uZXMKTHVja25vdyxCYW50aGFyYSBBUE1DLFJpY2UsQ29tbW9uLEZBUSwyMDI2LTAyLTI1LDMyNTAuMTgsMjQwMC4wLDg1MC4xNzk5OTk5OTk5OTk4LDEuMzU0MjQxNjY2NjY2NjY2NixUcnVlLEZhbHNlLFJzLi9RdWludGFsLFRvbm5lcwpMdWNrbm93LEx1Y2tub3cgQVBNQyxSaWNlLENvbW1vbixGQVEsMjAyNi0wMy0yNyw0NzExLjMxLDQyMjcuMTEsNDg0LjIwMDAwMDAwMDAwMDcsMS4xMTQ1NDYzNDQ5MDIzMDk0LFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKTHVja25vdyxMdWNrbm93IEFQTUMsUmljZSxDb21tb24sRkFRLDIwMjYtMDMtMjAsNDcxNS45NSw0NjA5Ljg2LDEwNi4wOTAwMDAwMDAwMDAxNSwxLjAyMzAxMzcxNDA4MjQyMzQsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpMdWNrbm93LEx1Y2tub3cgQVBNQyxSaWNlLENvbW1vbixGQVEsMjAyNi0wMy0xOCw2MTE0LjEzLDM3MDcuMjUsMjQwNi44OCwxLjY0OTIzNTk1NjU3MTU4MjgsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpMdWNrbm93LEx1Y2tub3cgQVBNQyxSaWNlLENvbW1vbixGQVEsMjAyNi0wMy0wNiw0NjEzLjM3LDM5MTYuMzIsNjk3LjA0OTk5OTk5OTk5OTcsMS4xNzc5ODU5NjY0MTc0NTMsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpMdWNrbm93LEx1Y2tub3cgQVBNQyxSaWNlLENvbW1vbixGQVEsMjAyNi0wMi0yNyw0ODgzLjUxLDQ3MzQuNjYsMTQ4Ljg1MDAwMDAwMDAwMDM2LDEuMDMxNDM4MzcxNDk4NjkyOCxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkx1Y2tub3csTHVja25vdyBBUE1DLFJpY2UsT3RoZXIsRkFRLDIwMjYtMDItMjcsNzA5OC43Nyw1MTAwLjAsMTk5OC43NzAwMDAwMDAwMDA0LDEuMzkxOTE1Njg2Mjc0NTEsVHJ1ZSxGYWxzZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLEdoYXppYWJhZCBBUE1DLFBvdGF0byxPdGhlcixGQVEsMjAyNi0wMi0yNywxMTAxLjc4LDExMTAuMDIsLTguMjQwMDAwMDAwMDAwMDA5LDAuOTkyNTc2NzEwMzI5NTQzNixUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxIYXB1ciBBUE1DLFBvdGF0byxPdGhlcixGQVEsMjAyNi0wMy0zMSw2MTEuNDgsMTEwMC4wLC00ODguNTIsMC41NTU4OTA5MDkwOTA5MDkxLFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxIYXB1ciBBUE1DLFBvdGF0byxPdGhlcixGQVEsMjAyNi0wMy0zMCw4MDQuNjMsOTQ3LjQ1LC0xNDIuODIwMDAwMDAwMDAwMDUsMC44NDkyNTg1MzYwNzA1MDQ5LFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLEhhcHVyIEFQTUMsUG90YXRvLE90aGVyLEZBUSwyMDI2LTAzLTI5LDk1OC4wOCwxMDcwLjU3LC0xMTIuNDg5OTk5OTk5OTk5OSwwLjg5NDkyNTEzMzM0MDE4MzMsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpHaGF6aWFiYWQsSGFwdXIgQVBNQyxQb3RhdG8sT3RoZXIsRkFRLDIwMjYtMDMtMjgsOTE1Ljc3LDk3Ni4zNywtNjAuNjAwMDAwMDAwMDAwMDIsMC45Mzc5MzMzNjU0MjQ5OTI1LFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLEhhcHVyIEFQTUMsUG90YXRvLE90aGVyLEZBUSwyMDI2LTAzLTI3LDk3Ny40MywxMDEwLjgyLC0zMy4zOTAwMDAwMDAwMDAxLDAuOTY2OTY3NDEyNTk1NzE0MyxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxIYXB1ciBBUE1DLFBvdGF0byxPdGhlcixGQVEsMjAyNi0wMy0yNiw5ODUuODcsMTA5MS42OSwtMTA1LjgyMDAwMDAwMDAwMDA1LDAuOTAzMDY3NzIwNjg5OTM5NSxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxIYXB1ciBBUE1DLFBvdGF0byxPdGhlcixGQVEsMjAyNi0wMy0yNSw4NjMuMTgsOTQxLjU2LC03OC4zOCwwLjkxNjc1NTE3MjI2NzMwMTEsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpHaGF6aWFiYWQsSGFwdXIgQVBNQyxQb3RhdG8sT3RoZXIsRkFRLDIwMjYtMDMtMjQsOTEwLjAxLDk0OS40MSwtMzkuMzk5OTk5OTk5OTk5OTgsMC45NTg1MDA1NDI0NDIxNDgzLFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLEhhcHVyIEFQTUMsUG90YXRvLE90aGVyLEZBUSwyMDI2LTAzLTIzLDk3MC41LDEwMzYuNjQsLTY2LjE0MDAwMDAwMDAwMDEsMC45MzYxOTc3MTU2OTY4NjY3LFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLEhhcHVyIEFQTUMsUG90YXRvLE90aGVyLEZBUSwyMDI2LTAzLTIyLDgyMi4zLDEwNjEuOTUsLTIzOS42NTAwMDAwMDAwMDAxLDAuNzc0MzMwMjQxNTM2Nzk1NCxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxIYXB1ciBBUE1DLFBvdGF0byxPdGhlcixGQVEsMjAyNi0wMy0yMCw4MDYuNTgsOTI0LjI0LC0xMTcuNjU5OTk5OTk5OTk5OTcsMC44NzI2OTU0MDM3OTEyMjMxLFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLEhhcHVyIEFQTUMsUG90YXRvLE90aGVyLEZBUSwyMDI2LTAzLTAzLDkzMC4yOSw5NzguNDgsLTQ4LjE5MDAwMDAwMDAwMDA1NSwwLjk1MDc1MDE0MzA3OTA2MTQsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpHaGF6aWFiYWQsSGFwdXIgQVBNQyxQb3RhdG8sT3RoZXIsRkFRLDIwMjYtMDMtMDEsMTA0Mi4yNCwxMDUyLjk3LC0xMC43MzAwMDAwMDAwMDAwMTgsMC45ODk4MDk3NzYxNTY5NjU1LFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLEhhcHVyIEFQTUMsUG90YXRvLE90aGVyLEZBUSwyMDI2LTAyLTI3LDg3NS44OCw4MzYuMzIsMzkuNTU5OTk5OTk5OTk5OTQ1LDEuMDQ3MzAyNDY3OTU0ODQ5NyxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxNdXJhZG5hZ2FyIEFQTUMsUG90YXRvLE90aGVyLEZBUSwyMDI2LTAzLTMxLDg2OC42OCw4MDAuMCw2OC42Nzk5OTk5OTk5OTk5NSwxLjA4NTg1LFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxNdXJhZG5hZ2FyIEFQTUMsUG90YXRvLE90aGVyLEZBUSwyMDI2LTAzLTI2LDg1OS4xLDgwMC4wLDU5LjEwMDAwMDAwMDAwMDAyLDEuMDczODc1MDAwMDAwMDAwMSxUcnVlLEZhbHNlLFJzLi9RdWludGFsLFRvbm5lcwpHaGF6aWFiYWQsTm9pZGEgQVBNQyxQb3RhdG8sT3RoZXIsRkFRLDIwMjYtMDMtMzAsMTU5Ni4yMyw3MDAuMCw4OTYuMjMsMi4yODAzMjg1NzE0Mjg1NzE1LFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxOb2lkYSBBUE1DLFBvdGF0byxPdGhlcixGQVEsMjAyNi0wMy0yOSw5OTMuNjYsNzAwLjAsMjkzLjY1OTk5OTk5OTk5OTk3LDEuNDE5NTE0Mjg1NzE0Mjg1NyxUcnVlLEZhbHNlLFJzLi9RdWludGFsLFRvbm5lcwpHaGF6aWFiYWQsTm9pZGEgQVBNQyxQb3RhdG8sT3RoZXIsRkFRLDIwMjYtMDMtMjYsODMzLjIyLDcwMC4wLDEzMy4yMjAwMDAwMDAwMDAwMywxLjE5MDMxNDI4NTcxNDI4NTcsVHJ1ZSxGYWxzZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLE5vaWRhIEFQTUMsUG90YXRvLE90aGVyLEZBUSwyMDI2LTAzLTI1LDg1OS4yOCw4NTYuMjIsMy4wNTk5OTk5OTk5OTk5NDU0LDEuMDAzNTczODQ3ODQyODQ0LFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLE5vaWRhIEFQTUMsUG90YXRvLE90aGVyLEZBUSwyMDI2LTAzLTEyLDk5My40Nyw5MzMuMDksNjAuMzc5OTk5OTk5OTk5OTk1LDEuMDY0NzA5NzI3ODkzMzQzNSxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxOb2lkYSBBUE1DLFBvdGF0byxPdGhlcixGQVEsMjAyNi0wMi0yNyw5MDYuNzMsOTEyLjY4LC01Ljk0OTk5OTk5OTk5OTkzMiwwLjk5MzQ4MDczODA0NjE5MzcsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpLYW5wdXIsS2FucHVyKEdyYWluKSBBUE1DLFBvdGF0byxPdGhlcixGQVEsMjAyNi0wMi0yNyw2MDEuMjQsNjAwLjAsMS4yNDAwMDAwMDAwMDAwMDksMS4wMDIwNjY2NjY2NjY2NjY3LFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkx1Y2tub3csTHVja25vdyBBUE1DLFBvdGF0byxPdGhlcixGQVEsMjAyNi0wMi0yNyw3MDEuMDgsNzAwLjAsMS4wODAwMDAwMDAwMDAwNDEsMS4wMDE1NDI4NTcxNDI4NTczLFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCg=="
}

print("\n=== BOOTSTRAP SOURCE MODULES & REFERENCE DATA ===")
for dest_path, b64_str in b64_payloads.items():
    decoded = base64.b64decode(b64_str)
    dest_path.write_bytes(decoded)
    sha = hashlib.sha256(decoded).hexdigest()
    print(f"Bootstrapped: {dest_path} ({len(decoded):,} bytes | sha256: {sha})")

# Assert existence and non-empty sizes
for p in [src_dir / "market_identity.py", src_dir / "data_pipeline.py", src_dir / "ingest_agmarknet_arrivals.py", src_dir / "baselines.py"]:
    assert p.exists() and p.stat().st_size > 0, f"Critical failure: missing or empty {p}"

# Verify import of all expected production helpers from bootstrapped modules
from src.baselines import (
    compute_forecast_metrics,
    cluster_bootstrap_series,
    evaluate_model_slice,
    load_canonical_disputed_keys,
    evaluate_series_train_eligibility,
    score_series_origin,
    check_serving_recency,
    extract_target_vectorized,
    predict_series_vectorized
)
print("Successfully imported all required baseline helpers from bootstrapped src/baselines.py.")

# Execute small isolated bootstrap smoke test on synthetic fixture
print("Executing bootstrap fixture smoke test...")
_smoke_dates = pd.date_range("2024-01-01", "2025-06-25", freq="D")
_smoke_prices = 2000.0 + np.sin(np.arange(len(_smoke_dates)) / 10.0) * 100.0
_smoke_dts = _smoke_dates.to_numpy(dtype="datetime64[ns]")
_smoke_str = _smoke_dates.strftime("%Y-%m-%d").to_numpy()
_smoke_res = score_series_origin(
    dates_dt=_smoke_dts,
    dates_str=_smoke_str,
    prices=_smoke_prices,
    origin_dt="2025-05-15",
    fold_info={"fold": "F1", "train_end": "2024-12-31", "test_start": "2025-01-08", "test_end": "2025-06-30"},
    series_meta={"series_id": "smoke__s1", "district": "Ghaziabad", "market": "Ghaziabad", "commodity": "Wheat", "variety": "Dara", "grade": "FAQ"}
)
assert _smoke_res["is_eligible_origin"], "Bootstrap smoke test failed: origin was not eligible"
assert np.isfinite(_smoke_res["pred_naive"]), "Bootstrap smoke test failed: pred_naive not finite"
assert np.isfinite(_smoke_res["pred_ma"]), "Bootstrap smoke test failed: pred_ma not finite"

_smoke_eval_df = pd.DataFrame({
    "series_id": ["smoke__s1"] * 5,
    "target_price": [2000.0, 2050.0, 2100.0, 2150.0, 2200.0],
    "pred_model": [2010.0, 2040.0, 2110.0, 2140.0, 2210.0],
    "pred_naive": [2000.0, 2000.0, 2100.0, 2100.0, 2200.0]
})
_smoke_eval = evaluate_model_slice(_smoke_eval_df, model_col="pred_model", naive_col="pred_naive", n_resamples=50)
assert _smoke_eval["n_scored_origins"] == 5, "Bootstrap smoke test failed: n_scored_origins != 5"
assert np.isfinite(_smoke_eval["paired_naive_mae"]), "Bootstrap smoke test failed: paired_naive_mae not finite"
assert np.isclose(_smoke_eval["skill_vs_naive"], 1.0 - (_smoke_eval["mae"] / _smoke_eval["paired_naive_mae"]), atol=1e-12), "Bootstrap smoke test failed: skill tolerance"
print("Bootstrap fixture smoke test PASSED successfully.")

# Resolve dataset root in Kaggle environment
KAGGLE_INPUT_DIR = "/kaggle/input/mandi-prediction-audit-003b"
if not os.path.exists(KAGGLE_INPUT_DIR):
    KAGGLE_INPUT_DIR = "/kaggle/input/datasets/theankitchoubey/mandi-prediction-audit-003b"
if not os.path.exists(KAGGLE_INPUT_DIR):
    cands = glob.glob("/kaggle/input/**/mandi_5year_validated.csv", recursive=True)
    if cands:
        KAGGLE_INPUT_DIR = os.path.dirname(os.path.dirname(os.path.dirname(cands[0])))
    else:
        KAGGLE_INPUT_DIR = "."

print(f"\nResolved Dataset Payload Root: {KAGGLE_INPUT_DIR}")

def compute_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

raw_csv_files = sorted(glob.glob(os.path.join(KAGGLE_INPUT_DIR, "5 year dataset/*.csv")))
if not raw_csv_files:
    raw_csv_files = sorted(glob.glob("5 year dataset/*.csv"))
print(f"Found {len(raw_csv_files)} raw CSV files.")
for f in raw_csv_files:
    print(f"  RAW: {os.path.basename(f)} ({os.path.getsize(f):,} bytes)")


In [ ]:
# Cell 2: Pipeline v2 Execution and Panel v2 SHA-256 Gate Assertion
from src.data_pipeline import MandiDataPipeline

print("Loading accepted market aliases...")
df_alias_all = pd.read_csv(ref_dir / "market_aliases.csv")
df_alias_accepted = df_alias_all[df_alias_all["status"] == "ACCEPTED"].copy()
print(f"Loaded {len(df_alias_accepted)} accepted alias entries.")

print("\nExecuting MandiDataPipeline v2 on 5-year raw files...")
pipeline_v2 = MandiDataPipeline(
    input_dir=os.path.join(KAGGLE_INPUT_DIR, "5 year dataset"),
    output_dir=".",
    output_filename="mandi_5year_validated_v2.csv",
    manifest_filename="processing_manifest_v2.json",
    min_series_records=20,
    exclude_ambiguous_same_grade=True,
    pipeline_version="v2",
    alias_df=df_alias_accepted
)

df_v2, manifest_v2, out_csv_v2, out_manifest_v2 = pipeline_v2.run(raw_csv_files)
print(f"Pipeline v2 Complete! Clean rows: {len(df_v2):,}, unique series: {df_v2['series_id'].nunique():,}")

# CRITICAL SHA-256 INTEGRITY GATE ASSERTION
v2_sha256 = compute_sha256("mandi_5year_validated_v2.csv")
print(f"\nPanel v2 SHA-256: {v2_sha256}")
expected_sha256 = "851e718468d6b8c9fd07944535a462338e71aa105af25339fe9438b697088f31"
if v2_sha256 != expected_sha256:
    raise RuntimeError(
        f"PROVISIONAL PANEL V2 SHA-256 MISMATCH! Got {v2_sha256}, expected {expected_sha256}. STOPPING EXECUTION."
    )
print(">>> PANEL V2 INTEGRITY ASSERTION PASSED SUCCESSFULLY! SHA-256 CONFIRMED. <<<")

# Standardize date format to string YYYY-MM-DD and ensure numeric Modal_Price_Clean
df_v2["Arrival_Date"] = pd.to_datetime(df_v2["Arrival_Date"]).dt.strftime("%Y-%m-%d")
df_v2["Modal_Price_Clean"] = pd.to_numeric(df_v2["Modal_Price"], errors="coerce")

# Load v1 panel for reconciliation and diagnostics
val_csv_v1_path = os.path.join(KAGGLE_INPUT_DIR, "data/processed/mandi_5year_validated.csv")
df_v1 = pd.read_csv(val_csv_v1_path)
df_v1["Arrival_Date"] = pd.to_datetime(df_v1["Arrival_Date"]).dt.strftime("%Y-%m-%d")
df_v1["Modal_Price_Clean"] = pd.to_numeric(df_v1["Modal_Price"], errors="coerce")
print(f"Loaded v1 validated panel: {len(df_v1):,} rows across {df_v1['series_id'].nunique():,} series.")



In [ ]:
# Cell 3: Part B1 - Forensic Diagnostic Gates (Report Only)
print("=== EXECUTING PART B1: FORENSIC GATES ===")

# Load all raw observations for exact transition and classification forensics
raw_dfs = []
for f in raw_csv_files:
    raw_dfs.append(pd.read_csv(f))
df_raw_all = pd.concat(raw_dfs, ignore_index=True)
df_raw_all["Arrival_Date_dt"] = pd.to_datetime(df_raw_all["Arrival_Date"], format="%d/%m/%Y", errors="coerce")
df_raw_all["Arrival_Date_str"] = df_raw_all["Arrival_Date_dt"].dt.strftime("%Y-%m-%d")
DISTRICT_NORMALIZATION = {
    "gaziabaad": "Ghaziabad", "ghaziabad": "Ghaziabad", "gaziabad": "Ghaziabad",
    "jhansi": "Jhansi", "kanpur": "Kanpur", "kanpur nagar": "Kanpur",
    "lukhnow": "Lucknow", "lucknow": "Lucknow", "unnao": "Unnao"
}
df_raw_all["District_Norm"] = (
    df_raw_all["District"].astype(str).str.strip().str.lower().map(DISTRICT_NORMALIZATION)
    .fillna(df_raw_all["District"].astype(str).str.strip())
)
df_raw_all["Market_Raw"] = df_raw_all["Market"].astype(str).str.strip()
print(f"Total raw observations loaded: {len(df_raw_all):,}")

# -------------------------------------------------------------
# Gate B1.a: Alias Transition Forensics
# -------------------------------------------------------------
print("\n--- Gate B1.a: Alias Transition Forensics ---")
alias_pairs_to_eval = []

# Collect all pairs where market_raw != market_canonical
for _, r in df_alias_all.iterrows():
    m_raw = str(r["market_raw"]).strip()
    m_can = str(r["market_canonical"]).strip()
    dist = str(r["district"]).strip()
    if m_raw != m_can:
        alias_pairs_to_eval.append({
            "district": dist,
            "market_raw": m_raw,
            "market_canonical": m_can,
            "status": r["status"],
            "eval_type": "Accepted Alias Pair" if r["status"] == "ACCEPTED" else "Review Pair"
        })

# Special candidate pairs specified in prompt
# 1. Baripaal APMC vs Varipaal APMC in Kanpur
alias_pairs_to_eval.append({
    "district": "Kanpur",
    "market_raw": "Baripaal APMC",
    "market_canonical": "Varipaal APMC",
    "status": "SPECIAL_EVAL",
    "eval_type": "Special Comparison (Baripaal APMC vs Varipaal APMC)"
})
# 2. Jhansi Grain variants in Jhansi: Jhansi Grain APMC vs Jhansi (Grain) APMC
alias_pairs_to_eval.append({
    "district": "Jhansi",
    "market_raw": "Jhansi Grain APMC",
    "market_canonical": "Jhansi (Grain) APMC",
    "status": "SPECIAL_EVAL",
    "eval_type": "Special Comparison (Jhansi Grain APMC vs Jhansi (Grain) APMC)"
})

transition_records = []
for item in alias_pairs_to_eval:
    dist = item["district"]
    m_raw = item["market_raw"]
    m_can = item["market_canonical"]
    
    sub_raw = df_raw_all[(df_raw_all["District_Norm"] == dist) & (df_raw_all["Market_Raw"] == m_raw)]
    sub_can = df_raw_all[(df_raw_all["District_Norm"] == dist) & (df_raw_all["Market_Raw"] == m_can)]
    
    dates_raw = sub_raw["Arrival_Date_dt"].dropna().sort_values()
    dates_can = sub_can["Arrival_Date_dt"].dropna().sort_values()
    
    first_raw = dates_raw.iloc[0] if len(dates_raw) > 0 else pd.NaT
    last_raw = dates_raw.iloc[-1] if len(dates_raw) > 0 else pd.NaT
    first_can = dates_can.iloc[0] if len(dates_can) > 0 else pd.NaT
    last_can = dates_can.iloc[-1] if len(dates_can) > 0 else pd.NaT
    
    # Identify older vs newer by first_date
    if pd.notna(first_can) and pd.notna(first_raw):
        if first_can <= first_raw:
            older_name, newer_name = m_can, m_raw
            older_first, older_last = first_can, last_can
            newer_first, newer_last = first_raw, last_raw
        else:
            older_name, newer_name = m_raw, m_can
            older_first, older_last = first_raw, last_raw
            newer_first, newer_last = first_can, last_can
            
        gap_days = int((newer_first - older_last).days)
        set_raw_dates = set(dates_raw)
        set_can_dates = set(dates_can)
        overlap_dates = set_raw_dates.intersection(set_can_dates)
        overlap_days = len(overlap_dates)
        
        # Overlapping keys and price agreement
        merge_keys = ["Commodity", "Variety", "Grade", "Arrival_Date_dt"]
        merged_keys = pd.merge(
            sub_can[merge_keys + ["Modal_Price"]],
            sub_raw[merge_keys + ["Modal_Price"]],
            on=merge_keys,
            suffixes=("_can", "_raw")
        )
        n_overlapping_keys = len(merged_keys)
        if n_overlapping_keys > 0:
            p_match = (merged_keys["Modal_Price_can"] == merged_keys["Modal_Price_raw"]).sum()
            exact_price_agreement = round(float(p_match / n_overlapping_keys * 100.0), 2)
        else:
            exact_price_agreement = np.nan
    else:
        gap_days = np.nan
        overlap_days = 0
        n_overlapping_keys = 0
        exact_price_agreement = np.nan
        older_name, newer_name = m_can, m_raw
        older_first, older_last = first_can, last_can
        newer_first, newer_last = first_raw, last_raw
        
    transition_records.append({
        "district": dist,
        "market_raw": m_raw,
        "market_canonical": m_can,
        "eval_type": item["eval_type"],
        "status": item["status"],
        "older_name": older_name,
        "newer_name": newer_name,
        "older_first_date": older_first.strftime("%Y-%m-%d") if pd.notna(older_first) else "",
        "older_last_date": older_last.strftime("%Y-%m-%d") if pd.notna(older_last) else "",
        "newer_first_date": newer_first.strftime("%Y-%m-%d") if pd.notna(newer_first) else "",
        "newer_last_date": newer_last.strftime("%Y-%m-%d") if pd.notna(newer_last) else "",
        "gap_days": gap_days,
        "overlap_days": overlap_days,
        "overlapping_keys_count": n_overlapping_keys,
        "exact_price_agreement_pct": exact_price_agreement
    })

df_transitions = pd.DataFrame(transition_records)
df_transitions.to_csv("alias_transition_forensics.csv", index=False)
print(f"Exported alias_transition_forensics.csv ({len(df_transitions)} candidate pairs evaluated)")

# -------------------------------------------------------------
# Gate B1.b: Reclassify 1,895 Unmatched In-Window Excel Rows on v2
# -------------------------------------------------------------
print("\n--- Gate B1.b: Reclassify Unmatched In-Window Excel Rows ---")
from src.ingest_agmarknet_arrivals import load_arrivals_file
from src.market_identity import apply_market_canonicalization

excel_files = sorted(glob.glob(os.path.join(KAGGLE_INPUT_DIR, "raw_downloads/*.xlsx")))
if not excel_files:
    excel_files = sorted(glob.glob(os.path.join(KAGGLE_INPUT_DIR, "data/raw/arrivals/historical/raw_downloads/*.xlsx")))
if not excel_files:
    excel_files = sorted(glob.glob(f"{KAGGLE_INPUT_DIR}/**/raw_downloads/*.xlsx", recursive=True))

excel_dfs = [load_arrivals_file(f) for f in excel_files]
df_excel = pd.concat(excel_dfs, ignore_index=True)
df_excel["Arrival_Date"] = pd.to_datetime(df_excel["Arrival_Date"]).dt.strftime("%Y-%m-%d")
df_excel["Modal_Price_Clean"] = pd.to_numeric(df_excel["Modal_Price"], errors="coerce")

# Filter in-window: 2021-09-01 to 2026-09-30 (14,741 total rows)
df_excel_in_win = df_excel[(df_excel["Arrival_Date"] >= "2021-09-01") & (df_excel["Arrival_Date"] <= "2026-09-30")].copy()
print(f"Total Excel in-window rows: {len(df_excel_in_win):,} (Expected: 14,741)")

# Canonicalize Market in df_excel using accepted alias table
df_excel_canon = apply_market_canonicalization(df_excel_in_win, df_alias_accepted)

match_cols_excel = ["District_Std", "Market", "Commodity_Std", "Variety", "Grade", "Arrival_Date"]
match_cols_panel = ["District", "Market", "Commodity", "Variety", "Grade", "Arrival_Date"]

# Ensure string dtypes on both sides
for c_e, c_p in zip(match_cols_excel, match_cols_panel):
    df_excel_canon[c_e] = df_excel_canon[c_e].astype(str).str.strip()
    df_v2[c_p] = df_v2[c_p].astype(str).str.strip()

df_v2_match = df_v2[match_cols_panel].drop_duplicates()
merged_check = pd.merge(
    df_excel_canon,
    df_v2_match,
    left_on=match_cols_excel,
    right_on=match_cols_panel,
    how="left",
    indicator=True
)

df_unmatched_excel = merged_check[merged_check["_merge"] == "left_only"].copy()
print(f"Verified Unmatched Excel Rows count on v2: {len(df_unmatched_excel):,} (Expected: 1,895)")

# Build lookup structures from raw data and dropped series
df_raw_v2 = df_raw_all.copy()
df_raw_v2["Market_Canonical"] = df_raw_v2["Market_Raw"].map(
    dict(zip(df_alias_accepted["market_raw"], df_alias_accepted["market_canonical"]))
).fillna(df_raw_v2["Market_Raw"])
df_raw_v2["Commodity_Std"] = df_raw_v2["Commodity"].astype(str).str.strip()
df_raw_v2["Variety_Std"] = df_raw_v2["Variety"].astype(str).str.strip()
df_raw_v2["Grade_Std"] = df_raw_v2["Grade"].astype(str).str.strip()
df_raw_v2["series_id_v2"] = (
    df_raw_v2["District_Norm"] + "__" + 
    df_raw_v2["Market_Canonical"] + "__" + 
    df_raw_v2["Commodity_Std"] + "__" + 
    df_raw_v2["Variety_Std"] + "__" + 
    df_raw_v2["Grade_Std"]
)
raw_series_counts = df_raw_v2.groupby("series_id_v2").size().to_dict()
raw_key_set = set(
    zip(
        df_raw_v2["District_Norm"],
        df_raw_v2["Market_Canonical"],
        df_raw_v2["Commodity_Std"],
        df_raw_v2["Variety_Std"],
        df_raw_v2["Grade_Std"],
        df_raw_v2["Arrival_Date_str"]
    )
)
raw_dmcd_set = set(
    zip(
        df_raw_v2["District_Norm"],
        df_raw_v2["Market_Canonical"],
        df_raw_v2["Commodity_Std"],
        df_raw_v2["Arrival_Date_str"]
    )
)
raw_dcd_set = set(
    zip(
        df_raw_v2["District_Norm"],
        df_raw_v2["Commodity_Std"],
        df_raw_v2["Arrival_Date_str"]
    )
)

# Reclassification
reclassified_records = []
for _, r in df_unmatched_excel.iterrows():
    d = r["District_Std"]
    m = r["Market"]
    c = r["Commodity_Std"]
    v = r["Variety"]
    g = r["Grade"]
    dt = r["Arrival_Date"]
    sid = f"{d}__{m}__{c}__{v}__{g}"
    
    key = (d, m, c, v, g, dt)
    dmcd = (d, m, c, dt)
    dcd = (d, c, dt)
    
    if key in raw_key_set:
        cnt = raw_series_counts.get(sid, 0)
        if cnt < 20:
            category = "low_obs_dropped"
        else:
            category = "quarantined_strategy2"
    elif dmcd in raw_dmcd_set:
        category = "spelling_unmapped"
    elif dcd in raw_dcd_set:
        category = "other"
    else:
        category = "absent_from_raw"
        
    reclassified_records.append({
        "District": d,
        "Market": m,
        "Commodity": c,
        "Variety": v,
        "Grade": g,
        "Arrival_Date": dt,
        "Modal_Price": r["Modal_Price"],
        "category": category
    })

df_reclass = pd.DataFrame(reclassified_records)

# Grouped summaries by District and Crop
unmatched_summary_by_dist_crop = df_reclass.groupby(["District", "Commodity", "category"]).size().reset_index(name="count")
unmatched_summary_by_dist_crop.to_csv("excel_unmatched_reclassification_v2.csv", index=False)
print("Saved excel_unmatched_reclassification_v2.csv")

overall_category_counts = df_reclass["category"].value_counts().to_dict()
print("\nUnmatched Excel Rows Category Breakdown (Total = 1,895):")
for cat, cnt in overall_category_counts.items():
    print(f"  {cat:<25}: {cnt:>5,} ({cnt/len(df_reclass)*100:.2f}%)")

with open("excel_unmatched_summary_v2.json", "w") as f:
    json.dump({
        "total_unmatched_excel_rows": len(df_reclass),
        "category_counts": overall_category_counts,
        "district_breakdown": df_reclass.groupby(["District", "category"]).size().unstack(fill_value=0).to_dict(orient="index")
    }, f, indent=2)
print("Saved excel_unmatched_summary_v2.json")

# -------------------------------------------------------------
# Gate B1.c: Reconcile Tier A Training Observations (Cutoff 2025-12-18)
# -------------------------------------------------------------
print("\n--- Gate B1.c: Reconcile Tier A Training Observations (Cutoff 2025-12-18) ---")
train_cutoff = "2025-12-18"

# v1 Tier A calculation
df_v1_train = df_v1[df_v1["Arrival_Date"] <= train_cutoff].copy()
df_v1_train["Arrival_Date_dt"] = pd.to_datetime(df_v1_train["Arrival_Date"])
v1_series_metrics = []
for sid, grp in df_v1_train.groupby("series_id"):
    dts = sorted(grp["Arrival_Date_dt"].drop_duplicates().tolist())
    n = len(grp)
    span = (dts[-1] - dts[0]).days + 1
    cov = n / span if span > 0 else 0.0
    if n >= 60 and cov >= 0.3:
        v1_series_metrics.append({
            "series_id": sid,
            "n_obs_v1": n,
            "span_v1": span,
            "cov_v1": round(cov, 4)
        })
df_tier_a_v1 = pd.DataFrame(v1_series_metrics)
v1_tier_a_obs = int(df_tier_a_v1["n_obs_v1"].sum())
print(f"v1 Tier A Series: {len(df_tier_a_v1):,}, Total Observations: {v1_tier_a_obs:,} (Expected: 384,887)")

# v2 Tier A calculation
df_v2_train = df_v2[df_v2["Arrival_Date"] <= train_cutoff].copy()
df_v2_train["Arrival_Date_dt"] = pd.to_datetime(df_v2_train["Arrival_Date"])
v2_series_metrics = []
for sid, grp in df_v2_train.groupby("series_id"):
    dts = sorted(grp["Arrival_Date_dt"].drop_duplicates().tolist())
    n = len(grp)
    span = (dts[-1] - dts[0]).days + 1
    cov = n / span if span > 0 else 0.0
    if n >= 60 and cov >= 0.3:
        v2_series_metrics.append({
            "series_id": sid,
            "n_obs_v2": n,
            "span_v2": span,
            "cov_v2": round(cov, 4)
        })
df_tier_a_v2 = pd.DataFrame(v2_series_metrics)
v2_tier_a_obs = int(df_tier_a_v2["n_obs_v2"].sum())
print(f"v2 Tier A Series: {len(df_tier_a_v2):,}, Total Observations: {v2_tier_a_obs:,} (Expected: 404,498)")
obs_delta_total = v2_tier_a_obs - v1_tier_a_obs
print(f"Net Tier A Observation Gain on v2: +{obs_delta_total:,} rows")

# Reconcile sets
s_v1 = set(df_tier_a_v1["series_id"])
s_v2 = set(df_tier_a_v2["series_id"])

entering_series = df_tier_a_v2[df_tier_a_v2["series_id"].isin(s_v2 - s_v1)].copy()
leaving_series = df_tier_a_v1[df_tier_a_v1["series_id"].isin(s_v1 - s_v2)].copy()

merged_common = pd.merge(df_tier_a_v1, df_tier_a_v2, on="series_id", how="inner")
merged_common["delta_obs"] = merged_common["n_obs_v2"] - merged_common["n_obs_v1"]

print(f"Series entering Tier A on v2: {len(entering_series):,} (Total Obs: {entering_series['n_obs_v2'].sum():,})")
print(f"Series leaving Tier A on v2: {len(leaving_series):,} (Total Obs: {leaving_series['n_obs_v1'].sum():,})")
print(f"Common Series: {len(merged_common):,} (Common delta obs: {merged_common['delta_obs'].sum():,})")

reconciliation_summary = {
    "cutoff_date": train_cutoff,
    "v1_tier_a": {
        "series_count": len(df_tier_a_v1),
        "total_observations": v1_tier_a_obs
    },
    "v2_tier_a": {
        "series_count": len(df_tier_a_v2),
        "total_observations": v2_tier_a_obs
    },
    "net_observation_delta": obs_delta_total,
    "entering_series": {
        "count": len(entering_series),
        "observations": int(entering_series["n_obs_v2"].sum()),
        "series_list": entering_series["series_id"].tolist()
    },
    "leaving_series": {
        "count": len(leaving_series),
        "observations": int(leaving_series["n_obs_v1"].sum()),
        "series_list": leaving_series["series_id"].tolist()
    },
    "common_series": {
        "count": len(merged_common),
        "delta_observations": int(merged_common["delta_obs"].sum())
    }
}
with open("tier_a_reconciliation_v1_vs_v2.json", "w") as f:
    json.dump(reconciliation_summary, f, indent=2)
merged_common.sort_values("delta_obs", ascending=False).to_csv("tier_a_series_deltas.csv", index=False)
print("Saved tier_a_reconciliation_v1_vs_v2.json and tier_a_series_deltas.csv")

# -------------------------------------------------------------
# Gate B1.d: Fractional Prices Diagnostics (Post 2026-03)
# -------------------------------------------------------------
print("\n--- Gate B1.d: Fractional Prices Diagnostics ---")
df_v2["is_fractional"] = (df_v2["Modal_Price_Clean"] % 1 != 0)
df_v2["decimal_part"] = np.round(df_v2["Modal_Price_Clean"] - np.floor(df_v2["Modal_Price_Clean"]), 4)
df_v2["year_month"] = df_v2["Arrival_Date"].str[:7]

# 1. Month-by-month fractional share across all 61 months
monthly_frac = df_v2.groupby("year_month").agg(
    total_obs=("Modal_Price_Clean", "count"),
    frac_obs=("is_fractional", "sum")
).reset_index()
monthly_frac["frac_share_pct"] = round(monthly_frac["frac_obs"] / monthly_frac["total_obs"] * 100.0, 2)
first_month_gt_5pct = monthly_frac[monthly_frac["frac_share_pct"] > 5.0].iloc[0]["year_month"] if np.any(monthly_frac["frac_share_pct"] > 5.0) else "None"
print(f"First calendar month where fractional share exceeds 5%: {first_month_gt_5pct}")

# 2. Decimal part distribution for 2026-03 to 2026-09
df_v2_post_mar = df_v2[df_v2["Arrival_Date"] >= "2026-03-01"].copy()
decimal_distribution = df_v2_post_mar["decimal_part"].value_counts().head(20).reset_index()
decimal_distribution.columns = ["decimal_part", "count"]
decimal_distribution["share_pct"] = round(decimal_distribution["count"] / len(df_v2_post_mar) * 100.0, 2)
decimal_distribution.to_csv("fractional_part_distribution.csv", index=False)
print("Saved fractional_part_distribution.csv (Top 10 decimals post 2026-03):")
print(decimal_distribution.head(10).to_string(index=False))

# 3. Adjacent-day integer test for same series
print("\nChecking adjacent-day integer prices for fractional series...")
df_v2_post_mar.sort_values(["series_id", "Arrival_Date"], inplace=True)
df_v2_post_mar["Arrival_Date_dt"] = pd.to_datetime(df_v2_post_mar["Arrival_Date"])
df_v2_post_mar["prev_dt"] = df_v2_post_mar.groupby("series_id")["Arrival_Date_dt"].shift(1)
df_v2_post_mar["next_dt"] = df_v2_post_mar.groupby("series_id")["Arrival_Date_dt"].shift(-1)
df_v2_post_mar["prev_price"] = df_v2_post_mar.groupby("series_id")["Modal_Price_Clean"].shift(1)
df_v2_post_mar["next_price"] = df_v2_post_mar.groupby("series_id")["Modal_Price_Clean"].shift(-1)

df_frac_rows = df_v2_post_mar[df_v2_post_mar["is_fractional"]].copy()
has_adj_prev = (df_frac_rows["Arrival_Date_dt"] - df_frac_rows["prev_dt"]).dt.days == 1
has_adj_next = (df_frac_rows["next_dt"] - df_frac_rows["Arrival_Date_dt"]).dt.days == 1

prev_is_int = (df_frac_rows["prev_price"] % 1 == 0) & has_adj_prev
next_is_int = (df_frac_rows["next_price"] % 1 == 0) & has_adj_next

has_any_adj = has_adj_prev | has_adj_next
any_adj_is_int = prev_is_int | next_is_int

pct_has_adj = round(has_any_adj.mean() * 100.0, 2)
pct_adj_int = round((any_adj_is_int[has_any_adj]).mean() * 100.0, 2) if has_any_adj.sum() > 0 else np.nan
print(f"Fractional rows with adjacent day (t +/- 1): {has_any_adj.sum():,} / {len(df_frac_rows):,} ({pct_has_adj}%)")
print(f"Share of those adjacent day observations that are exact integers: {pct_adj_int}%")

# 4. Commodity median price ratio before vs after 2026-03
df_pre = df_v2[df_v2["Arrival_Date"] < "2026-03-01"].groupby("Commodity")["Modal_Price_Clean"].median()
df_post = df_v2[df_v2["Arrival_Date"] >= "2026-03-01"].groupby("Commodity")["Modal_Price_Clean"].median()
comm_ratio = pd.DataFrame({"median_pre": df_pre, "median_post": df_post}).dropna()
comm_ratio["ratio_post_to_pre"] = round(comm_ratio["median_post"] / comm_ratio["median_pre"], 4)

frac_diagnostics = {
    "first_month_fractional_gt_5pct": first_month_gt_5pct,
    "fractional_rows_post_2026_03": int(df_v2_post_mar["is_fractional"].sum()),
    "total_rows_post_2026_03": len(df_v2_post_mar),
    "fractional_share_post_2026_03_pct": round(df_v2_post_mar["is_fractional"].mean() * 100.0, 2),
    "adjacent_day_test": {
        "fractional_rows_with_adjacent_obs_pct": pct_has_adj,
        "adjacent_obs_is_integer_pct": pct_adj_int
    },
    "commodity_median_ratio_summary": {
        "mean_ratio": round(float(comm_ratio["ratio_post_to_pre"].mean()), 4),
        "median_ratio": round(float(comm_ratio["ratio_post_to_pre"].median()), 4),
        "min_ratio": round(float(comm_ratio["ratio_post_to_pre"].min()), 4),
        "max_ratio": round(float(comm_ratio["ratio_post_to_pre"].max()), 4)
    }
}
with open("fractional_pricing_diagnostics.json", "w") as f:
    json.dump(frac_diagnostics, f, indent=2)
print("Saved fractional_pricing_diagnostics.json")

# -------------------------------------------------------------
# Gate B1.e: Monthly Reporting Depth Diagnostics
# -------------------------------------------------------------
print("\n--- Gate B1.e: Monthly Reporting Depth Diagnostics ---")
monthly_depth_records = []
for ym, grp in df_v2.groupby("year_month"):
    monthly_depth_records.append({
        "year_month": ym,
        "total_rows": len(grp),
        "active_markets_ghaziabad": grp[grp["District"] == "Ghaziabad"]["Market"].nunique(),
        "active_markets_jhansi": grp[grp["District"] == "Jhansi"]["Market"].nunique(),
        "active_markets_kanpur": grp[grp["District"] == "Kanpur"]["Market"].nunique(),
        "active_markets_lucknow": grp[grp["District"] == "Lucknow"]["Market"].nunique(),
        "active_markets_unnao": grp[grp["District"] == "Unnao"]["Market"].nunique(),
        "total_active_markets": grp["Market"].nunique(),
        "unique_commodities": grp["Commodity"].nunique(),
        "unique_varieties": grp["Variety"].nunique(),
        "unique_series": grp["series_id"].nunique()
    })
df_depth = pd.DataFrame(monthly_depth_records).sort_values("year_month")
df_depth.to_csv("monthly_reporting_depth_diagnostics.csv", index=False)
print(f"Exported monthly_reporting_depth_diagnostics.csv ({len(df_depth)} months)")
print(df_depth.tail(12).to_string(index=False))



In [ ]:
# Cell 4: Part B2 - Baselines Scorecard Engine across 4 Folds
from src.baselines import (
    compute_forecast_metrics,
    cluster_bootstrap_series,
    evaluate_model_slice,
    load_canonical_disputed_keys,
    evaluate_series_train_eligibility,
    score_series_origin
)

print("=== EXECUTING PART B2: BASELINES SCORECARD ENGINE ===")

FOLDS = [
    {
        "fold": "F1",
        "train_end": "2024-12-31",
        "test_start": "2025-01-08",
        "test_end": "2025-06-30"
    },
    {
        "fold": "F2",
        "train_end": "2025-06-30",
        "test_start": "2025-07-08",
        "test_end": "2025-10-31"
    },
    {
        "fold": "F3",
        "train_end": "2025-12-18",
        "test_start": "2025-12-26",
        "test_end": "2026-03-31"
    },
    {
        "fold": "F4",
        "train_end": "2026-03-31",
        "test_start": "2026-04-08",
        "test_end": "2026-09-30"
    }
]

FOCUS_CROPS = [
    "Wheat",
    "Potato",
    "Onion",
    "Tomato",
    "Paddy(Dhan)(Common)",
    "Rice"
]

# -------------------------------------------------------------
# Fail-Closed Canonical Disputed Keys Loading (Task MP-AUDIT-007 / 008)
# -------------------------------------------------------------
diff_77_candidates = [
    out005_dir / "diff_77_deep_analysis.csv",
    Path("/kaggle/input/mandi-prediction-audit-003b/diff_77_deep_analysis.csv"),
    Path("diff_77_deep_analysis.csv")
]
diff_77_path = None
for cp in diff_77_candidates:
    if cp.exists():
        diff_77_path = cp
        break

alias_candidates = [
    ref_dir / "market_aliases.csv",
    Path("/kaggle/input/mandi-prediction-audit-003b/market_aliases.csv"),
    Path("data/reference/market_aliases.csv"),
    Path("market_aliases.csv")
]
alias_path = None
for ap in alias_candidates:
    if ap.exists():
        alias_path = ap
        break

if diff_77_path is None:
    raise FileNotFoundError("CRITICAL AUDIT FAILURE: diff_77_deep_analysis.csv required but not found.")
if alias_path is None:
    raise FileNotFoundError("CRITICAL AUDIT FAILURE: market_aliases.csv required but not found.")

set_77_canon_keys, df_diff_77_canon = load_canonical_disputed_keys(
    disputed_csv_path=diff_77_path,
    aliases=alias_path,
    expected_rows=77
)
print(f"Verified and loaded {len(set_77_canon_keys)} canonical disputed keys (fail-closed check passed).")
df_diff_77_canon.to_csv("canonical_disputed_keys_audit.csv", index=False)
print("Saved canonical_disputed_keys_audit.csv")

# Pre-parse panel v2 for rapid vector access
df_v2["Arrival_Date_dt"] = pd.to_datetime(df_v2["Arrival_Date"])
df_v2_sorted = df_v2.sort_values(["series_id", "Arrival_Date_dt"]).reset_index(drop=True)

# Build per-series arrays dictionary
print("Indexing series observations...")
series_dict = {}
for sid, grp in df_v2_sorted.groupby("series_id"):
    dts = grp["Arrival_Date_dt"].to_numpy()
    dts_str = grp["Arrival_Date"].to_numpy()
    prices = grp["Modal_Price_Clean"].to_numpy(dtype=float)
    first_row = grp.iloc[0]
    series_dict[sid] = {
        "dates_dt": dts,
        "dates_str": dts_str,
        "prices": prices,
        "district": str(first_row["District"]),
        "market": str(first_row["Market"]),
        "commodity": str(first_row["Commodity"]),
        "variety": str(first_row["Variety"]),
        "grade": str(first_row["Grade"])
    }

print(f"Indexed {len(series_dict):,} series.")

# -------------------------------------------------------------
# Origin Evaluation Loop across 4 Folds Calling Tested Scoring Helper
# -------------------------------------------------------------
evaluated_origin_records = []
stage_counts = {
    "total_raw_test_window_observations": 0,
    "dropped_serving_recency_lt_3": 0,
    "dropped_no_target_in_window": 0,
    "dropped_target_after_fold_end": 0,
    "evaluated_test_origins": 0,
    "scope_a_excluded_count": 0,
    "scope_b_affected_or_ineligible_count": 0
}

for fold_info in FOLDS:
    fold_name = fold_info["fold"]
    tr_end = fold_info["train_end"]
    te_start = pd.Timestamp(fold_info["test_start"])
    te_end = pd.Timestamp(fold_info["test_end"])

    print(f"\n--- Profiling Eligibility for Fold {fold_name} (Train <= {fold_info['train_end']}) ---")

    tier_a_set = set()
    tier_b_set = set()

    for sid, sdata in series_dict.items():
        is_tier_a, is_tier_b = evaluate_series_train_eligibility(
            dates_dt=sdata["dates_dt"],
            train_end_dt=tr_end,
            min_obs=60,
            min_cov=0.3,
            tier_b_min_obs=120
        )
        if is_tier_a:
            tier_a_set.add(sid)
        if is_tier_b:
            tier_b_set.add(sid)

    print(f"Fold {fold_name}: Tier A series = {len(tier_a_set):,}, Tier B series = {len(tier_b_set):,}")

    all_eligible = tier_a_set.union(tier_b_set)

    for sid in sorted(all_eligible):
        sdata = series_dict[sid]
        series_meta = {
            "series_id": sid,
            "district": sdata["district"],
            "market": sdata["market"],
            "commodity": sdata["commodity"],
            "variety": sdata["variety"],
            "grade": sdata["grade"],
            "is_tier_a": (sid in tier_a_set),
            "is_tier_b": (sid in tier_b_set)
        }

        test_obs_indices = np.where((sdata["dates_dt"] >= te_start) & (sdata["dates_dt"] <= te_end))[0]
        stage_counts["total_raw_test_window_observations"] += len(test_obs_indices)

        for idx in test_obs_indices:
            origin_dt = sdata["dates_dt"][idx]

            res = score_series_origin(
                dates_dt=sdata["dates_dt"],
                dates_str=sdata["dates_str"],
                prices=sdata["prices"],
                origin_dt=origin_dt,
                fold_info=fold_info,
                series_meta=series_meta,
                canonical_disputed_keys=set_77_canon_keys
            )

            if not res["is_eligible_origin"]:
                reason = res.get("drop_reason")
                if reason == "dropped_serving_recency_lt_3":
                    stage_counts["dropped_serving_recency_lt_3"] += 1
                elif reason == "dropped_no_target_in_window":
                    stage_counts["dropped_no_target_in_window"] += 1
                elif reason == "dropped_target_after_fold_end":
                    stage_counts["dropped_target_after_fold_end"] += 1
                continue

            stage_counts["evaluated_test_origins"] += 1
            if res["in_77_scope_a"]:
                stage_counts["scope_a_excluded_count"] += 1
            if not res["scope_b_eligible"]:
                stage_counts["scope_b_affected_or_ineligible_count"] += 1

            evaluated_origin_records.append(res)

df_origins = pd.DataFrame(evaluated_origin_records)
print(f"\nTotal evaluated origin instances: {len(df_origins):,}")

# Save per-stage exclusion counts JSON
with open("exclusion_stage_counts.json", "w") as f:
    json.dump(stage_counts, f, indent=2)
print("Saved exclusion_stage_counts.json")

# Export predictions archives
df_origins.head(5000).to_csv("baseline_predictions_sample.csv", index=False)
print("Saved baseline_predictions_sample.csv (first 5,000 rows retained for backward compatibility).")
df_origins.to_csv("baseline_predictions_all_folds.csv", index=False)
print("Saved baseline_predictions_all_folds.csv (complete all-fold evaluated predictions archive).")
print("Note: The prior 5,000-row sample only contained F1 origins and is insufficient for independent full-fold reproduction.")

# Models for standard scorecard evaluation
MODELS = [
    ("Naive", "pred_naive"),
    ("Moving Average", "pred_ma"),
    ("Seasonal Naive", "pred_seasonal_naive"),
    ("Drift", "pred_drift")
]

# -------------------------------------------------------------
# Table 1: Fold x Tier Scorecard
# -------------------------------------------------------------
print("\n--- Generating Table 1: Fold x Tier Scorecard ---")
t1_rows = []

for fold_name in ["F1", "F2", "F3", "F4", "Overall"]:
    df_fold = df_origins if fold_name == "Overall" else df_origins[df_origins["fold"] == fold_name]

    for tier_name in ["Tier A", "Tier B"]:
        df_tier = df_fold[df_fold["is_tier_a"]] if tier_name == "Tier A" else df_fold[df_fold["is_tier_b"]]

        for model_label, model_col in MODELS:
            res = evaluate_model_slice(df_tier, model_col=model_col, naive_col="pred_naive")
            row = {
                "fold": fold_name,
                "tier": tier_name,
                "model": model_label,
                **res
            }
            t1_rows.append(row)

# Assert skill identity for Table 1
for r in t1_rows:
    if np.isfinite(r.get("paired_naive_mae", np.nan)) and r["paired_naive_mae"] > 0 and np.isfinite(r.get("mae", np.nan)):
        assert np.isclose(r["skill_vs_naive"], 1.0 - (r["mae"] / r["paired_naive_mae"]), atol=1e-4), f"Table 1 skill mismatch: {r}"

df_t1 = pd.DataFrame(t1_rows)
df_t1.to_csv("scorecard_fold_x_tier.csv", index=False)
df_t1.to_csv(scorecards_dir / "scorecard_fold_x_tier.csv", index=False)
print("Saved scorecard_fold_x_tier.csv")
print(df_t1[["fold", "tier", "model", "n_origins", "n_series", "mae", "mae_ci_lower", "mae_ci_upper", "skill_vs_naive", "low_n_flag"]].head(20).to_string())

# -------------------------------------------------------------
# Table 2: Fold x Focus Crop Scorecard (Tier A)
# -------------------------------------------------------------
print("\n--- Generating Table 2: Fold x Focus Crop Scorecard (Tier A) ---")
t2_rows = []

df_tier_a = df_origins[df_origins["is_tier_a"]]

for fold_name in ["F1", "F2", "F3", "F4", "Overall"]:
    df_fold = df_tier_a if fold_name == "Overall" else df_tier_a[df_tier_a["fold"] == fold_name]

    for crop in FOCUS_CROPS:
        df_crop = df_fold[df_fold["commodity"] == crop]

        for model_label, model_col in MODELS:
            res = evaluate_model_slice(df_crop, model_col=model_col, naive_col="pred_naive")
            row = {
                "fold": fold_name,
                "crop": crop,
                "model": model_label,
                **res
            }
            t2_rows.append(row)

# Assert skill identity for Table 2
for r in t2_rows:
    if np.isfinite(r.get("paired_naive_mae", np.nan)) and r["paired_naive_mae"] > 0 and np.isfinite(r.get("mae", np.nan)):
        assert np.isclose(r["skill_vs_naive"], 1.0 - (r["mae"] / r["paired_naive_mae"]), atol=1e-4), f"Table 2 skill mismatch: {r}"

df_t2 = pd.DataFrame(t2_rows)
df_t2.to_csv("scorecard_fold_x_focus_crop.csv", index=False)
df_t2.to_csv(scorecards_dir / "scorecard_fold_x_focus_crop.csv", index=False)
print("Saved scorecard_fold_x_focus_crop.csv")
print(df_t2[["fold", "crop", "model", "n_origins", "n_series", "mae", "mae_ci_lower", "mae_ci_upper", "skill_vs_naive", "low_n_flag"]].head(24).to_string())

# -------------------------------------------------------------
# Table 3: Sensitivities Scorecard (Tier A)
# -------------------------------------------------------------
print("\n--- Generating Table 3: Sensitivities Scorecard (Tier A) ---")
t3_rows = []

for fold_name in ["F1", "F2", "F3", "F4", "Overall"]:
    df_fold = df_tier_a if fold_name == "Overall" else df_tier_a[df_tier_a["fold"] == fold_name]

    sensitivities = [
        ("Exact T+7 Subset", df_fold[df_fold["exact_t7"]], "standard"),
        ("Scope (a): Excluding 77 Disputed Keys (Origin/Target Filter)", df_fold[~df_fold["in_77_scope_a"]], "standard"),
        ("Scope (b): Excising 77 Disputed History & Target", df_fold[df_fold["scope_b_eligible"]], "scope_b"),
        ("Target Move > 2% Only (Diagnostic Only)", df_fold[df_fold["target_move_gt_2pct"]], "standard")
    ]

    for sens_name, df_sens, mode in sensitivities:
        if mode == "scope_b":
            # For Scope B, evaluate all 4 models using their counterfactual clean history predictions
            scope_b_models = [
                ("Naive", "pred_naive_scope_b"),
                ("Moving Average", "pred_ma_scope_b"),
                ("Seasonal Naive", "pred_seasonal_scope_b"),
                ("Drift", "pred_drift_scope_b")
            ]
            for model_label, model_col in scope_b_models:
                res = evaluate_model_slice(df_sens, model_col=model_col, naive_col="pred_naive_scope_b")
                row = {
                    "fold": fold_name,
                    "sensitivity": sens_name,
                    "model": model_label,
                    **res
                }
                t3_rows.append(row)
        else:
            for model_label, model_col in MODELS:
                res = evaluate_model_slice(df_sens, model_col=model_col, naive_col="pred_naive")
                row = {
                    "fold": fold_name,
                    "sensitivity": sens_name,
                    "model": model_label,
                    **res
                }
                t3_rows.append(row)

# Assert skill identity for Table 3
for r in t3_rows:
    if np.isfinite(r.get("paired_naive_mae", np.nan)) and r["paired_naive_mae"] > 0 and np.isfinite(r.get("mae", np.nan)):
        assert np.isclose(r["skill_vs_naive"], 1.0 - (r["mae"] / r["paired_naive_mae"]), atol=1e-4), f"Table 3 skill mismatch: {r}"

df_t3 = pd.DataFrame(t3_rows)
df_t3.to_csv("scorecard_sensitivities.csv", index=False)
df_t3.to_csv(scorecards_dir / "scorecard_sensitivities.csv", index=False)
print("Saved scorecard_sensitivities.csv")
print(df_t3[["fold", "sensitivity", "model", "n_origins", "n_series", "mae", "mae_ci_lower", "mae_ci_upper", "skill_vs_naive", "low_n_flag"]].head(24).to_string())

# -------------------------------------------------------------
# Scope B Diagnostics Export (Fold x Crop Breakdown)
# -------------------------------------------------------------
print("\n--- Generating Scope B Diagnostics by Fold and Crop ---")
scope_b_diag_rows = []
for fold_name in ["F1", "F2", "F3", "F4", "Overall"]:
    df_f = df_tier_a if fold_name == "Overall" else df_tier_a[df_tier_a["fold"] == fold_name]
    for crop in ["ALL"] + FOCUS_CROPS:
        df_c = df_f if crop == "ALL" else df_f[df_f["commodity"] == crop]
        n_tot = len(df_c)
        if n_tot == 0:
            continue
        n_disputed_orig = int(df_c["is_disputed_origin"].sum())
        n_disputed_tgt = int(df_c["is_disputed_target"].sum())
        n_scope_a = int(df_c["in_77_scope_a"].sum())
        n_hist_removed = int(df_c["scope_b_has_removed_history"].sum())
        n_scope_b_eligible = int(df_c["scope_b_eligible"].sum())
        n_recency_failures = int((df_c["scope_b_ineligible_reason"] == "recency_failure_after_removal").sum())
        n_ma_changed = int(df_c["ma_changed_scope_b"].sum())
        n_seasonal_changed = int(df_c["seasonal_changed_scope_b"].sum())
        n_drift_changed = int(df_c["drift_changed_scope_b"].sum())

        scope_b_diag_rows.append({
            "fold": fold_name,
            "crop": crop,
            "total_origins": n_tot,
            "disputed_origin_count": n_disputed_orig,
            "disputed_target_count": n_disputed_tgt,
            "scope_a_excluded_count": n_scope_a,
            "scope_b_has_removed_history_count": n_hist_removed,
            "scope_b_recency_failure_count": n_recency_failures,
            "scope_b_eligible_count": n_scope_b_eligible,
            "scope_b_ma_changed_count": n_ma_changed,
            "scope_b_ma_finite_changed_count": int(df_c["ma_finite_price_changed"].sum()),
            "scope_b_ma_finite_to_nan_count": int(df_c["ma_finite_to_nan"].sum()),
            "scope_b_ma_nan_to_finite_count": int(df_c["ma_nan_to_finite"].sum()),
            "scope_b_seasonal_changed_count": n_seasonal_changed,
            "scope_b_seasonal_finite_changed_count": int(df_c["seasonal_finite_price_changed"].sum()),
            "scope_b_seasonal_finite_to_nan_count": int(df_c["seasonal_finite_to_nan"].sum()),
            "scope_b_seasonal_nan_to_finite_count": int(df_c["seasonal_nan_to_finite"].sum()),
            "scope_b_drift_changed_count": n_drift_changed,
            "scope_b_drift_finite_changed_count": int(df_c["drift_finite_price_changed"].sum()),
            "scope_b_drift_finite_to_nan_count": int(df_c["drift_finite_to_nan"].sum()),
            "scope_b_drift_nan_to_finite_count": int(df_c["drift_nan_to_finite"].sum())
        })

df_scope_b_diag = pd.DataFrame(scope_b_diag_rows)
df_scope_b_diag.to_csv("scope_b_diagnostics_by_fold_crop.csv", index=False)
print("Saved scope_b_diagnostics_by_fold_crop.csv")


In [ ]:
# Cell 5: Final Output Artifact Verification & SHA-256 Manifest
print("=== VERIFYING GENERATED OUTPUT ARTIFACTS ===")

expected_outputs = [
    "mandi_5year_validated_v2.csv",
    "processing_manifest_v2.json",
    "alias_transition_forensics.csv",
    "excel_unmatched_reclassification_v2.csv",
    "excel_unmatched_summary_v2.json",
    "tier_a_reconciliation_v1_vs_v2.json",
    "tier_a_series_deltas.csv",
    "fractional_part_distribution.csv",
    "fractional_pricing_diagnostics.json",
    "monthly_reporting_depth_diagnostics.csv",
    "canonical_disputed_keys_audit.csv",
    "exclusion_stage_counts.json",
    "baseline_predictions_sample.csv",
    "baseline_predictions_all_folds.csv",
    "scorecard_fold_x_tier.csv",
    "scorecard_fold_x_focus_crop.csv",
    "scorecard_sensitivities.csv",
    "scope_b_diagnostics_by_fold_crop.csv"
]

manifest_records = []
for fname in expected_outputs:
    p = Path(fname)
    if p.exists():
        sz = p.stat().st_size
        sha = compute_sha256(str(p))
        n_lines = sum(1 for _ in open(p, "rb")) if fname.endswith(".csv") else None
        print(f"FILE: {fname:<42} | {sz:>10,} bytes | lines: {str(n_lines):<8} | sha256: {sha}")
        manifest_records.append({
            "filename": fname,
            "bytes": sz,
            "lines": n_lines,
            "sha256": sha
        })
    else:
        print(f"FILE MISSING: {fname}")

with open("kernel_execution_manifest.json", "w") as f:
    json.dump({
        "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "total_artifacts": len(manifest_records),
        "artifacts": manifest_records
    }, f, indent=2)
print("\nSaved kernel_execution_manifest.json.")
print(">>> TASK MP-FC-006 KAGGLE EXECUTION COMPLETE! <<<")

